In [1]:
# ============================================================
# TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
# CELL 1 — CANONICAL ARTIFACT LINEAGE DIAGNOSTIC
# ============================================================

from pathlib import Path
import json
import platform
import sys
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER")
print("CELL 1 — CANONICAL ARTIFACT LINEAGE DIAGNOSTIC")
print("=" * 80)

print(f"Python : {sys.version}")
print(f"OS     : {platform.system()} {platform.release()}")
print(f"CWD    : {Path.cwd().resolve()}")


# ------------------------------------------------------------
# 1. Resolve project root
# ------------------------------------------------------------

CWD = Path.cwd().resolve()

if CWD.name.lower() == "notebooks":
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = next(
        (
            p
            for p in [CWD, *CWD.parents]
            if (
                (p / "Dataset").is_dir()
                and
                (p / "Notebooks").is_dir()
            )
        ),
        None,
    )

assert PROJECT_ROOT is not None, (
    "Could not resolve project root."
)

DATASET_DIR = PROJECT_ROOT / "Dataset"
NOTEBOOK_DIR = PROJECT_ROOT / "Notebooks"
SCRATCH_ROOT = PROJECT_ROOT / "scratch_mastery_outputs"


print("\nResolved project:")
print(f"  PROJECT_ROOT : {PROJECT_ROOT}")
print(f"  DATASET_DIR  : {DATASET_DIR}")
print(f"  NOTEBOOK_DIR : {NOTEBOOK_DIR}")
print(f"  SCRATCH_ROOT : {SCRATCH_ROOT}")


# ------------------------------------------------------------
# 2. Load frozen Phase-0 registry
# ------------------------------------------------------------

PATH_REGISTRY_PATH = (
    SCRATCH_ROOT
    / "00_project_setup"
    / "path_registry.json"
)

assert PATH_REGISTRY_PATH.is_file(), (
    f"path_registry.json missing:\n{PATH_REGISTRY_PATH}"
)

with open(
    PATH_REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as f:
    path_registry = json.load(f)


REGISTRY_PROJECT_ROOT = Path(
    path_registry["project_root"]
).resolve()

assert (
    REGISTRY_PROJECT_ROOT
    ==
    PROJECT_ROOT.resolve()
), (
    "Project-root mismatch.\n"
    f"Registry : {REGISTRY_PROJECT_ROOT}\n"
    f"Current  : {PROJECT_ROOT.resolve()}"
)


SCRATCH_OUTPUT_ROOT = Path(
    path_registry["scratch_output_root"]
)

print("\nFrozen registry:")
print(f"  {PATH_REGISTRY_PATH}")
print(f"  Scratch root: {SCRATCH_OUTPUT_ROOT}")


# ------------------------------------------------------------
# 3. Expected Phase-1 canonical location
# ------------------------------------------------------------

EXPECTED_CANONICAL_DIR = (
    SCRATCH_OUTPUT_ROOT
    / "01_data_foundation"
    / "03_integrity"
    / "canonical"
)


EXPECTED_CANONICAL_FILES = {
    "responses": "responses.parquet",
    "turns": "turns.parquet",
    "sessions": "sessions.parquet",
    "objectives": "objectives.parquet",
}


print("\n" + "=" * 80)
print("EXPECTED CANONICAL LOCATION")
print("=" * 80)

print(
    f"  {EXPECTED_CANONICAL_DIR}"
)

print(
    f"  exists = "
    f"{EXPECTED_CANONICAL_DIR.exists()}"
)


# ------------------------------------------------------------
# 4. Direct expected-path audit
# ------------------------------------------------------------

expected_rows = []

for artifact, filename in (
    EXPECTED_CANONICAL_FILES.items()
):

    path = EXPECTED_CANONICAL_DIR / filename

    expected_rows.append({
        "artifact": artifact,
        "filename": filename,
        "expected_path": str(path),
        "exists": path.is_file(),
        "size_mb": (
            round(
                path.stat().st_size
                / (1024 ** 2),
                2,
            )
            if path.is_file()
            else None
        ),
    })


expected_audit = pd.DataFrame(
    expected_rows
)

display(expected_audit)


# ------------------------------------------------------------
# 5. Search entire scratch tree for canonical filenames
#
# IMPORTANT:
# This is diagnostic only.
# We are NOT automatically accepting another location.
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CANONICAL FILENAME SEARCH — SCRATCH TREE")
print("=" * 80)


search_rows = []

for artifact, filename in (
    EXPECTED_CANONICAL_FILES.items()
):

    matches = sorted(
        SCRATCH_OUTPUT_ROOT.rglob(filename)
    )

    for path in matches:

        path_str = str(path).lower()

        if any(
            token in path_str
            for token in [
                ".production_tmp",
                ".build.tmp",
                "__pycache__",
                ".venv",
                "venv",
            ]
        ):
            continue

        search_rows.append({
            "artifact": artifact,
            "filename": filename,
            "path": str(path),
            "size_mb": round(
                path.stat().st_size
                / (1024 ** 2),
                2,
            ),
        })


# IMPORTANT: always create columns
search_columns = [
    "artifact",
    "filename",
    "path",
    "size_mb",
]

canonical_search = pd.DataFrame(
    search_rows,
    columns=search_columns,
)


if canonical_search.empty:

    print(
        "NO canonical parquet files were found "
        "under scratch_mastery_outputs."
    )

else:

    display(
        canonical_search
        .sort_values(
            ["artifact", "path"]
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 6. Search entire project as secondary diagnostic
#
# This is NOT an automatic source substitution.
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CANONICAL FILENAME SEARCH — ENTIRE PROJECT")
print("=" * 80)


project_search_rows = []

for artifact, filename in (
    EXPECTED_CANONICAL_FILES.items()
):

    matches = sorted(
        PROJECT_ROOT.rglob(filename)
    )

    for path in matches:

        path_str = str(path).lower()

        if any(
            token in path_str
            for token in [
                "site-packages",
                "__pycache__",
                ".venv",
                "venv",
                ".production_tmp",
                ".build.tmp",
            ]
        ):
            continue

        project_search_rows.append({
            "artifact": artifact,
            "filename": filename,
            "path": str(path),
            "size_mb": round(
                path.stat().st_size
                / (1024 ** 2),
                2,
            ),
        })


project_columns = [
    "artifact",
    "filename",
    "path",
    "size_mb",
]

project_canonical_search = pd.DataFrame(
    project_search_rows,
    columns=project_columns,
)


if project_canonical_search.empty:

    print(
        "NO canonical parquet filenames found "
        "anywhere in the project."
    )

else:

    display(
        project_canonical_search
        .sort_values(
            ["artifact", "path"]
        )
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 7. Show current candidate artifacts separately
#
# These are NOT accepted as canonical.
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CANDIDATE ARTIFACTS — DIAGNOSTIC ONLY")
print("=" * 80)


candidate_paths = {
    "turns_candidate":
        DATASET_DIR / "turns_candidate.parquet",

    "sessions_candidate":
        DATASET_DIR / "sessions_candidate.parquet",
}


candidate_rows = []

for name, path in candidate_paths.items():

    candidate_rows.append({
        "artifact": name,
        "path": str(path),
        "exists": path.is_file(),
        "size_mb": (
            round(
                path.stat().st_size
                / (1024 ** 2),
                2,
            )
            if path.is_file()
            else None
        ),
    })


candidate_audit = pd.DataFrame(
    candidate_rows
)

display(candidate_audit)


# ------------------------------------------------------------
# 8. Final diagnostic status
# ------------------------------------------------------------

expected_missing = (
    expected_audit.loc[
        ~expected_audit["exists"],
        "artifact",
    ].tolist()
)


print("\n" + "=" * 80)
print("R0 CANONICAL DISCOVERY STATUS")
print("=" * 80)


if not expected_missing:

    canonical_paths = {
        artifact:
            EXPECTED_CANONICAL_DIR / filename
        for artifact, filename
        in EXPECTED_CANONICAL_FILES.items()
    }

    R0_CANONICAL_DISCOVERY_READY = True

    print(
        "STATUS: CANONICAL_ARTIFACTS_FOUND"
    )

    for name, path in canonical_paths.items():
        print(f"  {name:12s}: {path}")

else:

    canonical_paths = {}

    R0_CANONICAL_DISCOVERY_READY = False

    print(
        "STATUS: CANONICAL_ARTIFACTS_MISSING"
    )

    print("\nMissing:")
    for name in expected_missing:
        print(f"  - {name}.parquet")

    print(
        "\nDO NOT use *_candidate.parquet "
        "as a replacement."
    )


print(
    f"\nR0_CANONICAL_DISCOVERY_READY = "
    f"{R0_CANONICAL_DISCOVERY_READY}"
)

print("=" * 80)

TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
CELL 1 — CANONICAL ARTIFACT LINEAGE DIAGNOSTIC
Python : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
OS     : Windows 10
CWD    : D:\Competition\Trace-the-race-local\Notebooks

Resolved project:
  PROJECT_ROOT : D:\Competition\Trace-the-race-local
  DATASET_DIR  : D:\Competition\Trace-the-race-local\Dataset
  NOTEBOOK_DIR : D:\Competition\Trace-the-race-local\Notebooks
  SCRATCH_ROOT : D:\Competition\Trace-the-race-local\scratch_mastery_outputs

Frozen registry:
  D:\Competition\Trace-the-race-local\scratch_mastery_outputs\00_project_setup\path_registry.json
  Scratch root: D:\Competition\Trace-the-race-local\scratch_mastery_outputs

EXPECTED CANONICAL LOCATION
  D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical
  exists = True


,artifact,filename,expected_path,exists,size_mb
0,responses,responses.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,True,0.52
1,turns,turns.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,True,1132.62
2,sessions,sessions.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,True,2.79
3,objectives,objectives.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,True,0.03



CANONICAL FILENAME SEARCH — SCRATCH TREE


,artifact,filename,path,size_mb
0,objectives,objectives.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,0.03
1,responses,responses.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,0.52
2,sessions,sessions.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,2.79
3,turns,turns.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,1132.62



CANONICAL FILENAME SEARCH — ENTIRE PROJECT


,artifact,filename,path,size_mb
0,objectives,objectives.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,0.03
1,responses,responses.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,0.52
2,sessions,sessions.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,2.79
3,turns,turns.parquet,D:\Competition\Trace-the-race-local\scratch_ma...,1132.62



CANDIDATE ARTIFACTS — DIAGNOSTIC ONLY


,artifact,path,exists,size_mb
0,turns_candidate,D:\Competition\Trace-the-race-local\Dataset\tu...,False,None
1,sessions_candidate,D:\Competition\Trace-the-race-local\Dataset\se...,False,None



R0 CANONICAL DISCOVERY STATUS
STATUS: CANONICAL_ARTIFACTS_FOUND
  responses   : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\responses.parquet
  turns       : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\turns.parquet
  sessions    : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\sessions.parquet
  objectives  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\objectives.parquet

R0_CANONICAL_DISCOVERY_READY = True


In [2]:
# ============================================================
# CELL 2 — CANONICAL SCHEMA / SHAPE AUDIT
# ============================================================

def parquet_schema_summary(path: Path, name: str):
    df = pd.read_parquet(path)

    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    print(f"Path  : {path}")
    print(f"Shape : {df.shape}")

    print("\nColumns:")
    print(list(df.columns))

    print("\nDtypes:")
    print(df.dtypes.to_string())

    print("\nNull counts:")
    nulls = df.isna().sum()
    print(nulls[nulls > 0].to_string() if (nulls > 0).any() else "No nulls")

    print("\nHead:")
    display(df.head(3))

    return df


responses = parquet_schema_summary(
    canonical_paths["responses"],
    "responses.parquet"
)

turns = parquet_schema_summary(
    canonical_paths["turns"],
    "turns.parquet"
)

sessions = parquet_schema_summary(
    canonical_paths["sessions"],
    "sessions.parquet"
)

objectives = parquet_schema_summary(
    canonical_paths["objectives"],
    "objectives.parquet"
)


RESPONSES.PARQUET
Path  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\responses.parquet
Shape : (35072, 7)

Columns:
['response_id', 'session_id', 'objective_id_raw', 'objective_raw', 'objective_uid', 'target', 'fold']

Dtypes:
response_id          object
session_id           object
objective_id_raw     object
objective_raw        object
objective_uid        object
target              float64
fold                  int64

Null counts:
No nulls

Head:


,response_id,session_id,objective_id_raw,objective_raw,objective_uid,target,fold
0,aaaavsh,bcaufvc,dqibnvd,Knowing the value of each digit in numbers wit...,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,1.0,3
1,aaabhzi,eyutanf,eukmzxl,Adding and subtracting tens to a 2-digit number.,OBJ_62a540f8645b1b14b4e0f078d1530b9fb0f8bf9493...,1.0,2
2,aaahpnz,juptkxd,fjbqcsv,Comparing and ordering fractions by finding a ...,OBJ_f33453a0ac95ec9160140d6bf47f02a59604f583ca...,0.0,4



TURNS.PARQUET
Path  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\turns.parquet
Shape : (6139854, 52)

Columns:
['session_id', 'turn_uid', 'source_row_uid', 'turn_index', 'session_id_raw', 'utterance_id_raw', 'utterance_id', 'utterance_id_status', 'role_raw', 'role', 'role_status', 'role_issue_flag', 'content_raw', 'text_norm', 'contains_unclear_flag', 'empty_after_normalization_flag', 'timestamp_raw', 'timestamp', 'timestamp_status', 'timestamp_kind', 'timestamp_precision', 'timezone_status', 'timestamp_day_offset', 'timestamp_order_value', 'timestamp_rollover_flag', 'ordering_method', 'ordering_confidence', 'ordering_issue_flag', 'ordering_comparability', 'role_turn_index', 'relative_turn_position', 'previous_role', 'next_role', 'speaker_switch', 'time_since_previous_turn', 'elapsed_from_session_start', 'is_first_turn', 'is_last_turn', 'source_file_relative', 'source_row_index', 'file_sha256', 'raw_field_hash', 'content_hash'

,session_id,turn_uid,source_row_uid,turn_index,session_id_raw,utterance_id_raw,utterance_id,utterance_id_status,role_raw,role,...,content_hash,turn_uid_version,missing_session_id_flag,missing_utterance_id_flag,missing_role_flag,empty_content_flag,missing_timestamp_flag,duplicate_utterance_id_flag,natural_key_collision_flag,duplicate_raw_field_flag
0,aaaedit,b42a03fee3a9fcf6b5d34f033d42896d409c6ad3fa2ab8...,345f367256e99f6a79cd183db1ee716cac132302405c80...,0,aaaedit,0,0,VALID,tutor,tutor,...,c36b40652bc4e6a89a36d5fd0a79cacb49f113c865de86...,1.0,False,False,False,False,False,False,False,False
1,aaaedit,76182f86f777c2402798e5c6f15f66d08b97a0fbd6c0a0...,4715f653dd108b64f2907e6d6e5f395d9e15b050cb00b5...,1,aaaedit,1,1,VALID,background,background,...,239674fd86ecd7be8672a4f17e6de45a479c64086d40bb...,1.0,False,False,False,False,False,False,False,False
2,aaaedit,c84d92d0b88ce952b2baac597b244712acc01bf3baaabc...,7b26d2b47d0d19b819438dbb21b83b41e5ab07e1a8d6f5...,2,aaaedit,2,2,VALID,background,background,...,4487548212d8f2f4736ac097d1a3043ffa5f56459056b8...,1.0,False,False,False,False,False,False,False,False



SESSIONS.PARQUET
Path  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\sessions.parquet
Shape : (22821, 33)

Columns:
['session_id', 'source_file_relative', 'file_sha256', 'source_raw_row_count', 'n_turns', 'n_student_turns', 'n_tutor_turns', 'n_background_turns', 'n_unknown_roles', 'first_valid_timestamp', 'last_valid_timestamp', 'duration_seconds', 'duration_status', 'timestamp_issue_count', 'utterance_id_issue_count', 'ordering_issue_count', 'unknown_role_count', 'empty_content_count', 'ordering_method', 'ordering_confidence', 'ordering_comparability', 'fallback_order_used', 'timestamp_tie_count', 'timestamp_id_conflict', 'timestamp_source_conflict', 'id_source_conflict', 'midnight_rollover_count', 'ambiguous_order_flag', 'raw_transcript_hash', 'normalized_transcript_hash', 'quality_warning_count', 'exact_duplicate_group', 'exact_normalized_duplicate_group']

Dtypes:
session_id                           object
source_file_rela

,session_id,source_file_relative,file_sha256,source_raw_row_count,n_turns,n_student_turns,n_tutor_turns,n_background_turns,n_unknown_roles,first_valid_timestamp,...,timestamp_id_conflict,timestamp_source_conflict,id_source_conflict,midnight_rollover_count,ambiguous_order_flag,raw_transcript_hash,normalized_transcript_hash,quality_warning_count,exact_duplicate_group,exact_normalized_duplicate_group
0,aaaedit,aaaedit.csv,eed066ebe1c1ec2b68d4c8c1bc4afd080063f78c495d37...,254,254,114,136,4,0,0,...,False,False,False,0,False,c80b368477ac71b00bfd733022db5662ca5ed51f89a421...,ce673bb667dad83138f5bd1877b584308e981dc3623807...,0,None,None
1,aaaptjd,aaaptjd.csv,ce0834d60e9a31e7402cd97a3e3bd69300d6d94370bc89...,360,360,178,165,17,0,0,...,False,False,False,0,False,e888acdf6c26a15a22a798cf942a81d2cd1d07a5983db0...,ab4d18683c2ad5a5e7e061c7e3a2ed1443ac0de46148f2...,0,None,None
2,aabkeov,aabkeov.csv,e01c76f619e3a1684131b1f177f7a6fb0e24d89b3b24ff...,281,281,136,137,8,0,0,...,False,False,False,0,False,27eed4839606eddc05b463b28ee69da4f13945fc3ec606...,df6252310be3c89e0cf8426396b7e0f2ff09b340d3b4ec...,0,None,None



OBJECTIVES.PARQUET
Path  : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\objectives.parquet
Shape : (398, 5)

Columns:
['objective_uid', 'objective_raw', 'objective_safe_norm', 'response_count', 'session_count']

Dtypes:
objective_uid          object
objective_raw          object
objective_safe_norm    object
response_count          int64
session_count           int64

Null counts:
No nulls

Head:


,objective_uid,objective_raw,objective_safe_norm,response_count,session_count
0,OBJ_eea26e4c30a7c92a7c980aa26f4ac58396e6120944...,Adding 4-digit numbers using the column method,Adding 4-digit numbers using the column method,2,2
1,OBJ_60c3037f37b88b0df1a9397498ea6df7b1e4b74a10...,Adding and subtracting 10s or 100s.,Adding and subtracting 10s or 100s.,75,75
2,OBJ_1c111c9b707c3058d0d5739fca4abdd76e906c4d7f...,Adding and subtracting amounts of money to giv...,Adding and subtracting amounts of money to giv...,76,76


In [3]:
# ============================================================
# CELL 3 — R0 RELATIONAL / COVERAGE CONTRACT
# ============================================================

print("=" * 80)
print("TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER")
print("CELL 3 — RELATIONAL / COVERAGE CONTRACT")
print("=" * 80)


# ------------------------------------------------------------
# 1. Required canonical columns
# ------------------------------------------------------------

REQUIRED_COLUMNS = {
    "responses": [
        "response_id",
        "session_id",
        "objective_id_raw",
        "objective_raw",
        "objective_uid",
        "fold",
    ],
    "turns": [
        "session_id",
        "turn_uid",
        "turn_index",
        "role",
        "text_norm",
    ],
    "sessions": [
        "session_id",
    ],
    "objectives": [
        "objective_uid",
        "objective_raw",
        "objective_safe_norm",
    ],
}


# ------------------------------------------------------------
# 2. Column contract
# ------------------------------------------------------------

contract_rows = []

for artifact_name, required_cols in REQUIRED_COLUMNS.items():

    df = {
        "responses": responses,
        "turns": turns,
        "sessions": sessions,
        "objectives": objectives,
    }[artifact_name]

    missing = [
        col for col in required_cols
        if col not in df.columns
    ]

    contract_rows.append({
        "artifact": artifact_name,
        "required_columns": len(required_cols),
        "missing_columns": missing,
        "passed": len(missing) == 0,
    })


contract_audit = pd.DataFrame(contract_rows)

display(contract_audit)

assert contract_audit["passed"].all(), (
    "R0 column contract failed."
)


# ------------------------------------------------------------
# 3. Basic identity checks
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("IDENTITY CHECKS")
print("=" * 80)


response_id_unique = (
    responses["response_id"].is_unique
)

session_id_unique = (
    sessions["session_id"].is_unique
)

turn_uid_unique = (
    turns["turn_uid"].is_unique
)

objective_uid_unique = (
    objectives["objective_uid"].is_unique
)


identity_audit = pd.DataFrame([
    {
        "check": "response_id unique",
        "observed": int(responses["response_id"].nunique()),
        "expected": len(responses),
        "passed": response_id_unique,
    },
    {
        "check": "session_id unique",
        "observed": int(sessions["session_id"].nunique()),
        "expected": len(sessions),
        "passed": session_id_unique,
    },
    {
        "check": "turn_uid unique",
        "observed": int(turns["turn_uid"].nunique()),
        "expected": len(turns),
        "passed": turn_uid_unique,
    },
    {
        "check": "objective_uid unique",
        "observed": int(objectives["objective_uid"].nunique()),
        "expected": len(objectives),
        "passed": objective_uid_unique,
    },
])

display(identity_audit)

assert identity_audit["passed"].all(), (
    "R0 identity contract failed."
)


# ------------------------------------------------------------
# 4. Response → session coverage
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESPONSE → SESSION COVERAGE")
print("=" * 80)


response_sessions = set(
    responses["session_id"].dropna()
)

canonical_sessions = set(
    sessions["session_id"].dropna()
)

turn_sessions = set(
    turns["session_id"].dropna()
)


missing_response_sessions = (
    response_sessions - canonical_sessions
)

missing_turn_sessions = (
    response_sessions - turn_sessions
)


response_session_coverage = pd.DataFrame([
    {
        "check": "Every response session exists in sessions",
        "observed": len(missing_response_sessions),
        "expected": 0,
        "passed": len(missing_response_sessions) == 0,
    },
    {
        "check": "Every response session exists in turns",
        "observed": len(missing_turn_sessions),
        "expected": 0,
        "passed": len(missing_turn_sessions) == 0,
    },
])

display(response_session_coverage)

assert response_session_coverage["passed"].all(), (
    "Response → session coverage failed."
)


# ------------------------------------------------------------
# 5. Response → objective coverage
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESPONSE → OBJECTIVE COVERAGE")
print("=" * 80)


response_objectives = set(
    responses["objective_uid"].dropna()
)

canonical_objectives = set(
    objectives["objective_uid"].dropna()
)

missing_objectives = (
    response_objectives - canonical_objectives
)


response_objective_coverage = pd.DataFrame([
    {
        "check": "Every response objective exists in objective catalogue",
        "observed": len(missing_objectives),
        "expected": 0,
        "passed": len(missing_objectives) == 0,
    },
])

display(response_objective_coverage)

assert response_objective_coverage["passed"].all(), (
    "Response → objective coverage failed."
)


# ------------------------------------------------------------
# 6. Objective identity consistency
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OBJECTIVE IDENTITY CONSISTENCY")
print("=" * 80)


objective_lookup = (
    objectives[
        [
            "objective_uid",
            "objective_raw",
            "objective_safe_norm",
        ]
    ]
    .drop_duplicates("objective_uid")
    .set_index("objective_uid")
)


response_objective_view = (
    responses[
        [
            "response_id",
            "objective_uid",
            "objective_raw",
        ]
    ]
    .copy()
)


response_objective_view = (
    response_objective_view
    .join(
        objective_lookup[
            ["objective_raw", "objective_safe_norm"]
        ],
        on="objective_uid",
        rsuffix="_catalogue",
    )
)


raw_objective_mismatch = (
    response_objective_view["objective_raw"]
    !=
    response_objective_view["objective_raw_catalogue"]
)


raw_objective_mismatch_count = int(
    raw_objective_mismatch.sum()
)


objective_identity_audit = pd.DataFrame([
    {
        "check": "Response objective_raw matches canonical objective_raw",
        "observed": raw_objective_mismatch_count,
        "expected": 0,
        "passed": raw_objective_mismatch_count == 0,
    }
])

display(objective_identity_audit)

assert raw_objective_mismatch_count == 0, (
    "Response objective text does not match canonical objective catalogue."
)


# ------------------------------------------------------------
# 7. Response cardinality contract
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESPONSE CARDINALITY")
print("=" * 80)


response_count = len(responses)

assert response_count == 35072, (
    f"Unexpected response population: {response_count}"
)


response_cardinality_audit = pd.DataFrame([
    {
        "artifact": "responses",
        "rows": response_count,
        "expected": 35072,
        "passed": response_count == 35072,
    },
    {
        "artifact": "sessions",
        "rows": len(sessions),
        "expected": 22821,
        "passed": len(sessions) == 22821,
    },
    {
        "artifact": "turns",
        "rows": len(turns),
        "expected": 6139854,
        "passed": len(turns) == 6139854,
    },
    {
        "artifact": "objectives",
        "rows": len(objectives),
        "expected": 398,
        "passed": len(objectives) == 398,
    },
])

display(response_cardinality_audit)

assert response_cardinality_audit["passed"].all(), (
    "Canonical population changed unexpectedly."
)


# ------------------------------------------------------------
# 8. Target isolation contract
#
# R0 retrieval artifacts must never carry target.
# Canonical responses are allowed to contain target.
# ------------------------------------------------------------

TARGET_COLUMN = "target"

assert TARGET_COLUMN in responses.columns, (
    "Canonical responses unexpectedly lacks target."
)

print("\nTarget exists in canonical responses: True")
print("R0 output target policy: EXCLUDED")


# ------------------------------------------------------------
# 9. Final Cell-3 gate
# ------------------------------------------------------------

R0_RELATIONAL_CONTRACT_READY = all([
    contract_audit["passed"].all(),
    identity_audit["passed"].all(),
    response_session_coverage["passed"].all(),
    response_objective_coverage["passed"].all(),
    objective_identity_audit["passed"].all(),
    response_cardinality_audit["passed"].all(),
    TARGET_COLUMN in responses.columns,
])


print("\n" + "=" * 80)
print("R0 CELL 3 STATUS")
print("=" * 80)

print(
    f"Required schema contract       : "
    f"{contract_audit['passed'].all()}"
)

print(
    f"Identity contract              : "
    f"{identity_audit['passed'].all()}"
)

print(
    f"Response → session coverage    : "
    f"{response_session_coverage['passed'].all()}"
)

print(
    f"Response → objective coverage  : "
    f"{response_objective_coverage['passed'].all()}"
)

print(
    f"Objective identity consistency : "
    f"{objective_identity_audit['passed'].all()}"
)

print(
    f"Population contract            : "
    f"{response_cardinality_audit['passed'].all()}"
)

print(
    f"Target isolation policy        : "
    f"{TARGET_COLUMN in responses.columns}"
)

print(
    f"\nR0_RELATIONAL_CONTRACT_READY = "
    f"{R0_RELATIONAL_CONTRACT_READY}"
)

assert R0_RELATIONAL_CONTRACT_READY, (
    "R0 relational contract failed."
)

print("=" * 80)

TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
CELL 3 — RELATIONAL / COVERAGE CONTRACT


,artifact,required_columns,missing_columns,passed
0,responses,6,[],True
1,turns,5,[],True
2,sessions,1,[],True
3,objectives,3,[],True



IDENTITY CHECKS


,check,observed,expected,passed
0,response_id unique,35072,35072,True
1,session_id unique,22821,22821,True
2,turn_uid unique,6139854,6139854,True
3,objective_uid unique,398,398,True



RESPONSE → SESSION COVERAGE


...


OBJECTIVE IDENTITY CONSISTENCY


,check,observed,expected,passed
0,Response objective_raw matches canonical objec...,0,0,True



RESPONSE CARDINALITY


,artifact,rows,expected,passed
0,responses,35072,35072,True
1,sessions,22821,22821,True
2,turns,6139854,6139854,True
3,objectives,398,398,True



Target exists in canonical responses: True
R0 output target policy: EXCLUDED

R0 CELL 3 STATUS
Required schema contract       : True
Identity contract              : True
Response → session coverage    : True
Response → objective coverage  : True
Objective identity consistency : True
Population contract            : True
Target isolation policy        : True

R0_RELATIONAL_CONTRACT_READY = True


In [4]:
# ============================================================
# CELL 4 — R0 RETRIEVAL QUERY BUILD + AUDIT
# ============================================================

print("=" * 80)
print("TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER")
print("CELL 4 — RETRIEVAL QUERY BUILD + AUDIT")
print("=" * 80)


# ------------------------------------------------------------
# 0. Preconditions
# ------------------------------------------------------------

assert R0_RELATIONAL_CONTRACT_READY is True, (
    "Cell 3 contract is not ready."
)


# ------------------------------------------------------------
# 1. Define the exact R0 query schema
#
# IMPORTANT:
# - target is deliberately excluded
# - fold is retained only as evaluation/provenance metadata
# - no target-derived statistics are introduced
# ------------------------------------------------------------

R0_QUERY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "objective_raw",
    "objective_safe_norm",
    "fold",
]


# ------------------------------------------------------------
# 2. Build canonical objective lookup
# ------------------------------------------------------------

objective_lookup_r0 = (
    objectives[
        [
            "objective_uid",
            "objective_raw",
            "objective_safe_norm",
        ]
    ]
    .drop_duplicates(subset=["objective_uid"])
    .copy()
)

assert objective_lookup_r0["objective_uid"].is_unique, (
    "Objective lookup is not one-row-per-objective_uid."
)


# ------------------------------------------------------------
# 3. Build response → objective retrieval query table
# ------------------------------------------------------------

retrieval_queries = (
    responses[
        [
            "response_id",
            "session_id",
            "objective_uid",
            "fold",
        ]
    ]
    .merge(
        objective_lookup_r0,
        on="objective_uid",
        how="left",
        validate="many_to_one",
        sort=False,
    )
)


# ------------------------------------------------------------
# 4. Reorder columns into frozen R0 schema
# ------------------------------------------------------------

retrieval_queries = retrieval_queries[
    R0_QUERY_COLUMNS
].copy()


# ------------------------------------------------------------
# 5. Deterministic ordering
#
# response_id is the immutable response-level identity.
# Sorting explicitly prevents input-row order from becoming
# an accidental part of the R0 artifact.
# ------------------------------------------------------------

retrieval_queries = (
    retrieval_queries
    .sort_values(
        by=["response_id"],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Basic shape / identity audit
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("R0 RETRIEVAL QUERY SHAPE / IDENTITY")
print("=" * 80)

query_rows = len(retrieval_queries)

print(f"Rows              : {query_rows}")
print(f"Columns           : {len(retrieval_queries.columns)}")
print(f"Expected rows     : 35072")
print(f"Expected columns  : {len(R0_QUERY_COLUMNS)}")


assert query_rows == 35072, (
    f"R0 retrieval query population changed: {query_rows}"
)

assert list(retrieval_queries.columns) == R0_QUERY_COLUMNS, (
    "R0 retrieval query schema/order mismatch."
)

assert retrieval_queries["response_id"].is_unique, (
    "response_id is not unique in retrieval_queries."
)


# ------------------------------------------------------------
# 7. Null / blank text audit
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RETRIEVAL TEXT AUDIT")
print("=" * 80)


text_audit = pd.DataFrame([
    {
        "field": "objective_raw",
        "null_rows": int(
            retrieval_queries["objective_raw"].isna().sum()
        ),
        "blank_rows": int(
            retrieval_queries["objective_raw"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),
    },
    {
        "field": "objective_safe_norm",
        "null_rows": int(
            retrieval_queries["objective_safe_norm"].isna().sum()
        ),
        "blank_rows": int(
            retrieval_queries["objective_safe_norm"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),
    },
])

display(text_audit)

assert (
    text_audit["null_rows"].sum() == 0
), "Null retrieval text detected."

assert (
    text_audit["blank_rows"].sum() == 0
), "Blank retrieval text detected."


# ------------------------------------------------------------
# 8. Objective cardinality audit
# ------------------------------------------------------------

query_objective_count = (
    retrieval_queries["objective_uid"].nunique()
)

print("\n" + "=" * 80)
print("OBJECTIVE COVERAGE")
print("=" * 80)

print(
    f"Unique objective_uid : {query_objective_count}"
)
print(
    f"Expected             : 398"
)

assert query_objective_count == 398, (
    "Retrieval query objective population changed."
)


# ------------------------------------------------------------
# 9. Fold integrity audit
#
# Fold is metadata only. It must exactly preserve the
# canonical response assignment.
# ------------------------------------------------------------

fold_mismatch = (
    retrieval_queries
    .merge(
        responses[
            [
                "response_id",
                "fold",
            ]
        ],
        on="response_id",
        how="left",
        validate="one_to_one",
        suffixes=("_r0", "_canonical"),
    )
)

fold_mismatch_count = int(
    (
        fold_mismatch["fold_r0"]
        !=
        fold_mismatch["fold_canonical"]
    ).sum()
)

print("\n" + "=" * 80)
print("FOLD METADATA INTEGRITY")
print("=" * 80)

print(f"Fold mismatches : {fold_mismatch_count}")
print("Expected        : 0")

assert fold_mismatch_count == 0, (
    "R0 fold metadata differs from canonical response assignment."
)


# ------------------------------------------------------------
# 10. Target isolation audit
#
# Explicitly reject target or target-derived columns.
# ------------------------------------------------------------

PROHIBITED_R0_QUERY_COLUMNS = {
    "target",
    "label",
    "prediction",
    "positive_rate",
    "target_mean",
    "target_count",
    "positive_count",
    "negative_count",
    "accuracy",
    "auc",
    "log_loss",
}


query_columns_lower = {
    str(col).lower()
    for col in retrieval_queries.columns
}

prohibited_present = sorted(
    query_columns_lower
    & PROHIBITED_R0_QUERY_COLUMNS
)

print("\n" + "=" * 80)
print("TARGET / LABEL ISOLATION")
print("=" * 80)

print(
    f"Prohibited columns present : {prohibited_present}"
)

assert prohibited_present == [], (
    "Target/label-derived field leaked into R0 retrieval queries."
)


# ------------------------------------------------------------
# 11. Session coverage audit
# ------------------------------------------------------------

query_sessions = set(
    retrieval_queries["session_id"]
)

canonical_session_ids = set(
    sessions["session_id"]
)

missing_query_sessions = (
    query_sessions - canonical_session_ids
)

print("\n" + "=" * 80)
print("SESSION COVERAGE")
print("=" * 80)

print(
    f"Unique query sessions : {len(query_sessions)}"
)

print(
    f"Missing canonical sessions : "
    f"{len(missing_query_sessions)}"
)

assert len(missing_query_sessions) == 0, (
    "R0 retrieval queries reference unknown sessions."
)


# ------------------------------------------------------------
# 12. Deterministic ordering audit
# ------------------------------------------------------------

response_ids_sorted = (
    retrieval_queries["response_id"]
    .tolist()
)

assert response_ids_sorted == sorted(response_ids_sorted), (
    "R0 retrieval queries are not deterministically ordered."
)


# ------------------------------------------------------------
# 13. Exact schema / dtype diagnostic
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("R0 QUERY SCHEMA")
print("=" * 80)

display(
    pd.DataFrame({
        "column": retrieval_queries.columns,
        "dtype": [
            str(retrieval_queries[col].dtype)
            for col in retrieval_queries.columns
        ],
    })
)


# ------------------------------------------------------------
# 14. Final R0 query audit table
# ------------------------------------------------------------

r0_query_checks = pd.DataFrame([
    {
        "check": "Exact row population",
        "observed": len(retrieval_queries),
        "expected": 35072,
        "passed": len(retrieval_queries) == 35072,
    },
    {
        "check": "Exact schema",
        "observed": list(retrieval_queries.columns),
        "expected": R0_QUERY_COLUMNS,
        "passed": list(retrieval_queries.columns) == R0_QUERY_COLUMNS,
    },
    {
        "check": "response_id unique",
        "observed": int(
            retrieval_queries["response_id"].nunique()
        ),
        "expected": 35072,
        "passed": retrieval_queries["response_id"].is_unique,
    },
    {
        "check": "Objective population",
        "observed": query_objective_count,
        "expected": 398,
        "passed": query_objective_count == 398,
    },
    {
        "check": "Retrieval text non-null/non-blank",
        "observed": int(
            text_audit["null_rows"].sum()
            +
            text_audit["blank_rows"].sum()
        ),
        "expected": 0,
        "passed": (
            text_audit["null_rows"].sum() == 0
            and
            text_audit["blank_rows"].sum() == 0
        ),
    },
    {
        "check": "Fold metadata preserved",
        "observed": fold_mismatch_count,
        "expected": 0,
        "passed": fold_mismatch_count == 0,
    },
    {
        "check": "Unknown sessions",
        "observed": len(missing_query_sessions),
        "expected": 0,
        "passed": len(missing_query_sessions) == 0,
    },
    {
        "check": "Target-derived columns",
        "observed": prohibited_present,
        "expected": [],
        "passed": prohibited_present == [],
    },
    {
        "check": "Deterministic response ordering",
        "observed": True,
        "expected": True,
        "passed": True,
    },
])

display(r0_query_checks)


# ------------------------------------------------------------
# 15. Final Cell-4 gate
# ------------------------------------------------------------

R0_QUERY_BUILD_READY = bool(
    r0_query_checks["passed"].all()
)

print("\n" + "=" * 80)
print("R0 CELL 4 STATUS")
print("=" * 80)

print(
    f"Retrieval query rows          : "
    f"{len(retrieval_queries)}"
)

print(
    f"Retrieval query schema        : "
    f"{len(retrieval_queries.columns)} fields"
)

print(
    f"Objective coverage            : "
    f"{query_objective_count}/398"
)

print(
    f"Target-derived fields         : "
    f"{len(prohibited_present)}"
)

print(
    f"R0_QUERY_BUILD_READY          : "
    f"{R0_QUERY_BUILD_READY}"
)

assert R0_QUERY_BUILD_READY, (
    "R0 retrieval query build failed."
)

print("=" * 80)

TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
CELL 4 — RETRIEVAL QUERY BUILD + AUDIT

R0 RETRIEVAL QUERY SHAPE / IDENTITY
Rows              : 35072
Columns           : 6
Expected rows     : 35072
Expected columns  : 6

RETRIEVAL TEXT AUDIT


,field,null_rows,blank_rows
0,objective_raw,0,0
1,objective_safe_norm,0,0



OBJECTIVE COVERAGE
Unique objective_uid : 398
Expected             : 398

FOLD METADATA INTEGRITY
Fold mismatches : 0
Expected        : 0

TARGET / LABEL ISOLATION
Prohibited columns present : []

SESSION COVERAGE
Unique query sessions : 22821
Missing canonical sessions : 0

R0 QUERY SCHEMA


,column,dtype
0,response_id,object
1,session_id,object
2,objective_uid,object
3,objective_raw,object
4,objective_safe_norm,object
5,fold,int64


,check,observed,expected,passed
0,Exact row population,35072,35072,True
1,Exact schema,"[response_id, session_id, objective_uid, objec...","[response_id, session_id, objective_uid, objec...",True
2,response_id unique,35072,35072,True
3,Objective population,398,398,True
4,Retrieval text non-null/non-blank,0,0,True
5,Fold metadata preserved,0,0,True
6,Unknown sessions,0,0,True
7,Target-derived columns,[],[],True
8,Deterministic response ordering,True,True,True



R0 CELL 4 STATUS
Retrieval query rows          : 35072
Retrieval query schema        : 6 fields
Objective coverage            : 398/398
Target-derived fields         : 0
R0_QUERY_BUILD_READY          : True


In [5]:
# ============================================================
# CELL 5 — SESSION TURN INDEX CONTRACT
# ============================================================

print("=" * 80)
print("TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER")
print("CELL 5 — SESSION TURN INDEX CONTRACT")
print("=" * 80)


# ------------------------------------------------------------
# 0. Preconditions
# ------------------------------------------------------------

assert R0_QUERY_BUILD_READY is True, (
    "Cell 4 retrieval query build is not ready."
)


# ------------------------------------------------------------
# 1. Exact fields required by R0 turn index
# ------------------------------------------------------------

R0_TURN_COLUMNS = [
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "text_norm",
    "relative_turn_position",
    "previous_role",
    "next_role",
    "speaker_switch",
    "time_since_previous_turn",
    "elapsed_from_session_start",
    "ordering_method",
    "ordering_confidence",
    "ordering_comparability",
]


missing_turn_columns = [
    col
    for col in R0_TURN_COLUMNS
    if col not in turns.columns
]

print("\nRequired R0 turn fields:")
print(R0_TURN_COLUMNS)

print("\nMissing:")
print(missing_turn_columns)

assert not missing_turn_columns, (
    f"Required R0 turn columns missing: {missing_turn_columns}"
)


# ------------------------------------------------------------
# 2. Exact population
# ------------------------------------------------------------

turn_rows = len(turns)

print("\n" + "=" * 80)
print("TURN POPULATION")
print("=" * 80)

print(f"Observed : {turn_rows}")
print(f"Expected : 6139854")

assert turn_rows == 6139854, (
    f"Unexpected turn population: {turn_rows}"
)


# ------------------------------------------------------------
# 3. Turn identity
# ------------------------------------------------------------

turn_uid_unique = turns["turn_uid"].is_unique

print("\n" + "=" * 80)
print("TURN IDENTITY")
print("=" * 80)

print(
    f"Unique turn_uid : "
    f"{turns['turn_uid'].nunique()}/{turn_rows}"
)

assert turn_uid_unique, (
    "turn_uid is not globally unique."
)


# ------------------------------------------------------------
# 4. Session coverage
# ------------------------------------------------------------

turn_session_ids = set(
    turns["session_id"].dropna()
)

canonical_session_ids = set(
    sessions["session_id"].dropna()
)

unknown_turn_sessions = (
    turn_session_ids - canonical_session_ids
)

print("\n" + "=" * 80)
print("TURN → SESSION COVERAGE")
print("=" * 80)

print(
    f"Turn sessions       : {len(turn_session_ids)}"
)

print(
    f"Canonical sessions  : {len(canonical_session_ids)}"
)

print(
    f"Unknown sessions    : {len(unknown_turn_sessions)}"
)

assert len(unknown_turn_sessions) == 0, (
    "Turns reference unknown canonical sessions."
)


# ------------------------------------------------------------
# 5. Required text audit
# ------------------------------------------------------------

raw_null = int(
    turns["content_raw"].isna().sum()
)

norm_null = int(
    turns["text_norm"].isna().sum()
)

raw_blank = int(
    turns["content_raw"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

norm_blank = int(
    turns["text_norm"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

text_audit_r0 = pd.DataFrame([
    {
        "field": "content_raw",
        "null_rows": raw_null,
        "blank_rows": raw_blank,
    },
    {
        "field": "text_norm",
        "null_rows": norm_null,
        "blank_rows": norm_blank,
    },
])

display(text_audit_r0)

assert norm_null == 0, (
    "text_norm contains null rows."
)

assert norm_blank == 0, (
    "text_norm contains blank rows."
)


# ------------------------------------------------------------
# 6. Role audit
# ------------------------------------------------------------

allowed_roles = {
    "student",
    "tutor",
    "background",
}

observed_roles = set(
    turns["role"].dropna().unique()
)

unexpected_roles = (
    observed_roles - allowed_roles
)

role_counts = (
    turns["role"]
    .value_counts(dropna=False)
    .rename_axis("role")
    .reset_index(name="rows")
)

print("\n" + "=" * 80)
print("ROLE AUDIT")
print("=" * 80)

display(role_counts)

print(
    f"Unexpected roles : {sorted(unexpected_roles)}"
)

assert len(unexpected_roles) == 0, (
    f"Unexpected role values detected: {unexpected_roles}"
)


# ------------------------------------------------------------
# 7. Within-session turn ordering
#
# Do NOT sort the full 6.1M-row frame unnecessarily.
# Canonical turns are already certified in session/turn order.
# We verify the existing order directly.
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("WITHIN-SESSION TURN ORDER")
print("=" * 80)

session_change = (
    turns["session_id"]
    .ne(turns["session_id"].shift())
)

session_start_indices = (
    session_change[session_change].index
)

session_end_indices = (
    list(session_start_indices[1:] - 1)
    + [len(turns) - 1]
)

session_start_indices = list(
    session_start_indices
)


# ------------------------------------------------------------
# Check each session's turn_index starts at 0 and increments
# by exactly one.
# ------------------------------------------------------------

turn_index_diff = (
    turns["turn_index"]
    .diff()
)

same_session = (
    turns["session_id"]
    ==
    turns["session_id"].shift()
)

bad_continuations = (
    same_session
    &
    (turn_index_diff != 1)
)

first_turn_bad = (
    turns.loc[
        session_start_indices,
        "turn_index"
    ]
    .ne(0)
)

ordering_failure_count = int(
    bad_continuations.sum()
    +
    first_turn_bad.sum()
)

print(
    f"Turn-index continuity failures : "
    f"{ordering_failure_count}"
)

print(
    "Expected                       : 0"
)

assert ordering_failure_count == 0, (
    "Within-session turn_index is not contiguous."
)


# ------------------------------------------------------------
# 8. First/last turn flags
# ------------------------------------------------------------

first_flag_count = int(
    turns.groupby("session_id")["is_first_turn"]
    .sum()
    .sub(1)
    .abs()
    .sum()
)

last_flag_count = int(
    turns.groupby("session_id")["is_last_turn"]
    .sum()
    .sub(1)
    .abs()
    .sum()
)

print("\n" + "=" * 80)
print("SESSION BOUNDARY FLAGS")
print("=" * 80)

print(
    f"First-turn flag failures : {first_flag_count}"
)

print(
    f"Last-turn flag failures  : {last_flag_count}"
)

assert first_flag_count == 0, (
    "Incorrect first-turn flag census."
)

assert last_flag_count == 0, (
    "Incorrect last-turn flag census."
)


# ------------------------------------------------------------
# 9. Relative position bounds
# ------------------------------------------------------------

relative_invalid = int(
    (
        turns["relative_turn_position"].lt(0)
        |
        turns["relative_turn_position"].gt(1)
        |
        turns["relative_turn_position"].isna()
    ).sum()
)

print("\n" + "=" * 80)
print("RELATIVE POSITION")
print("=" * 80)

print(
    f"Invalid relative positions : {relative_invalid}"
)

print("Expected                    : 0")

assert relative_invalid == 0, (
    "Invalid relative_turn_position values detected."
)


# ------------------------------------------------------------
# 10. Session census reconciliation
# ------------------------------------------------------------

turn_counts_from_turns = (
    turns
    .groupby("session_id", sort=False)
    .size()
    .rename("turn_count_from_turns")
)

session_census = (
    sessions[
        [
            "session_id",
            "n_turns",
        ]
    ]
    .set_index("session_id")
)

session_reconciliation = (
    session_census
    .join(turn_counts_from_turns, how="left")
)

session_reconciliation["turn_count_from_turns"] = (
    session_reconciliation["turn_count_from_turns"]
    .fillna(0)
    .astype("int64")
)

session_reconciliation_failures = int(
    (
        session_reconciliation["n_turns"]
        !=
        session_reconciliation["turn_count_from_turns"]
    ).sum()
)

print("\n" + "=" * 80)
print("SESSION TURN CENSUS")
print("=" * 80)

print(
    f"Session reconciliation failures : "
    f"{session_reconciliation_failures}"
)

print("Expected                        : 0")

assert session_reconciliation_failures == 0, (
    "Session n_turns does not reconcile with turns."
)


# ------------------------------------------------------------
# 11. Ordering metadata audit
# ------------------------------------------------------------

ordering_issue_rows = int(
    turns["ordering_issue_flag"].sum()
)

ordering_methods = (
    turns["ordering_method"]
    .value_counts(dropna=False)
    .to_dict()
)

ordering_confidence = (
    turns["ordering_confidence"]
    .value_counts(dropna=False)
    .to_dict()
)

ordering_comparability = (
    turns["ordering_comparability"]
    .value_counts(dropna=False)
    .to_dict()
)

print("\n" + "=" * 80)
print("ORDERING METADATA")
print("=" * 80)

print(
    f"Ordering issue rows : {ordering_issue_rows}"
)

print("\nOrdering methods:")
print(ordering_methods)

print("\nOrdering confidence:")
print(ordering_confidence)

print("\nOrdering comparability:")
print(ordering_comparability)


# ------------------------------------------------------------
# 12. Build lightweight audit dataframe
# ------------------------------------------------------------

r0_turn_contract_checks = pd.DataFrame([
    {
        "check": "Exact turn population",
        "observed": turn_rows,
        "expected": 6139854,
        "passed": turn_rows == 6139854,
    },
    {
        "check": "turn_uid globally unique",
        "observed": int(turns["turn_uid"].nunique()),
        "expected": 6139854,
        "passed": turn_uid_unique,
    },
    {
        "check": "Unknown turn sessions",
        "observed": len(unknown_turn_sessions),
        "expected": 0,
        "passed": len(unknown_turn_sessions) == 0,
    },
    {
        "check": "Normalized text null/blank",
        "observed": norm_null + norm_blank,
        "expected": 0,
        "passed": norm_null == 0 and norm_blank == 0,
    },
    {
        "check": "Unexpected roles",
        "observed": sorted(unexpected_roles),
        "expected": [],
        "passed": len(unexpected_roles) == 0,
    },
    {
        "check": "Turn-index continuity",
        "observed": ordering_failure_count,
        "expected": 0,
        "passed": ordering_failure_count == 0,
    },
    {
        "check": "First-turn flags",
        "observed": first_flag_count,
        "expected": 0,
        "passed": first_flag_count == 0,
    },
    {
        "check": "Last-turn flags",
        "observed": last_flag_count,
        "expected": 0,
        "passed": last_flag_count == 0,
    },
    {
        "check": "Relative position validity",
        "observed": relative_invalid,
        "expected": 0,
        "passed": relative_invalid == 0,
    },
    {
        "check": "Session turn census",
        "observed": session_reconciliation_failures,
        "expected": 0,
        "passed": session_reconciliation_failures == 0,
    },
])


display(r0_turn_contract_checks)


# ------------------------------------------------------------
# 13. Final Cell-5 gate
# ------------------------------------------------------------

R0_TURN_CONTRACT_READY = bool(
    r0_turn_contract_checks["passed"].all()
)

print("\n" + "=" * 80)
print("R0 CELL 5 STATUS")
print("=" * 80)

print(
    f"Turn rows                    : {turn_rows}"
)

print(
    f"Unique sessions in turns     : "
    f"{len(turn_session_ids)}"
)

print(
    f"Ordering failures            : "
    f"{ordering_failure_count}"
)

print(
    f"Text null/blank failures     : "
    f"{norm_null + norm_blank}"
)

print(
    f"Session census failures      : "
    f"{session_reconciliation_failures}"
)

print(
    f"R0_TURN_CONTRACT_READY       : "
    f"{R0_TURN_CONTRACT_READY}"
)

assert R0_TURN_CONTRACT_READY, (
    "R0 turn contract failed."
)

print("=" * 80)

TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
CELL 5 — SESSION TURN INDEX CONTRACT

Required R0 turn fields:
['session_id', 'turn_uid', 'turn_index', 'role', 'text_norm', 'relative_turn_position', 'previous_role', 'next_role', 'speaker_switch', 'time_since_previous_turn', 'elapsed_from_session_start', 'ordering_method', 'ordering_confidence', 'ordering_comparability']

Missing:
[]

TURN POPULATION
Observed : 6139854
Expected : 6139854

TURN IDENTITY
Unique turn_uid : 6139854/6139854

TURN → SESSION COVERAGE
Turn sessions       : 22821
Canonical sessions  : 22821
Unknown sessions    : 0


,field,null_rows,blank_rows
0,content_raw,0,0
1,text_norm,0,0



ROLE AUDIT


,role,rows
0,tutor,3196001
1,student,2697152
2,background,246701


Unexpected roles : []

WITHIN-SESSION TURN ORDER
Turn-index continuity failures : 0
Expected                       : 0

SESSION BOUNDARY FLAGS
First-turn flag failures : 0
Last-turn flag failures  : 0

RELATIVE POSITION
Invalid relative positions : 0
Expected                    : 0

SESSION TURN CENSUS
Session reconciliation failures : 0
Expected                        : 0

ORDERING METADATA
Ordering issue rows : 0

Ordering methods:
{'TIMESTAMP_PRIMARY': 6139854}

Ordering confidence:
{'HIGH': 6139854}

Ordering comparability:
{'PARTIALLY_COMPARABLE': 6137562, 'FULLY_COMPARABLE': 2292}


,check,observed,expected,passed
0,Exact turn population,6139854,6139854,True
1,turn_uid globally unique,6139854,6139854,True
2,Unknown turn sessions,0,0,True
3,Normalized text null/blank,0,0,True
4,Unexpected roles,[],[],True
5,Turn-index continuity,0,0,True
6,First-turn flags,0,0,True
7,Last-turn flags,0,0,True
8,Relative position validity,0,0,True
9,Session turn census,0,0,True



R0 CELL 5 STATUS
Turn rows                    : 6139854
Unique sessions in turns     : 22821
Ordering failures            : 0
Text null/blank failures     : 0
Session census failures      : 0
R0_TURN_CONTRACT_READY       : True


In [6]:
# ============================================================
# TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
# CELL 6 — BUILD RETRIEVAL QUERIES
# ============================================================

from pathlib import Path
import hashlib
import json
import gc

print("=" * 80)
print("TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER")
print("CELL 6 — BUILD RETRIEVAL QUERIES")
print("=" * 80)


# ------------------------------------------------------------
# 1. Preconditions
# ------------------------------------------------------------

assert R0_CANONICAL_DISCOVERY_READY is True
assert R0_RELATIONAL_CONTRACT_READY is True
assert R0_TURN_CONTRACT_READY is True


# ------------------------------------------------------------
# 2. R0 output directory
# ------------------------------------------------------------

R0_ROOT = (
    SCRATCH_OUTPUT_ROOT
    / "02_retrieval"
    / "R0_input"
)

R0_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 3. Exact retrieval-query schema
# ------------------------------------------------------------

QUERY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_raw",
    "objective_uid",
    "fold",
]


missing_query_columns = [
    col
    for col in QUERY_COLUMNS
    if col not in responses.columns
]

assert missing_query_columns == [], (
    f"Missing retrieval query columns: "
    f"{missing_query_columns}"
)


# ------------------------------------------------------------
# 4. Build label-blind query table
# ------------------------------------------------------------

retrieval_queries = (
    responses[
        QUERY_COLUMNS
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Target isolation
# ------------------------------------------------------------

PROHIBITED_QUERY_COLUMNS = {
    "target",
    "label",
    "prediction",
    "target_mean",
    "positive_rate",
    "positive_count",
    "negative_count",
}

query_leakage = sorted(
    set(retrieval_queries.columns)
    &
    PROHIBITED_QUERY_COLUMNS
)

assert query_leakage == [], (
    f"Target/evaluation leakage detected: "
    f"{query_leakage}"
)


# ------------------------------------------------------------
# 6. Identity checks
# ------------------------------------------------------------

assert retrieval_queries["response_id"].is_unique, (
    "response_id must be globally unique."
)

assert retrieval_queries["session_id"].notna().all(), (
    "session_id contains null values."
)

assert retrieval_queries["objective_uid"].notna().all(), (
    "objective_uid contains null values."
)

assert retrieval_queries["objective_raw"].notna().all(), (
    "objective_raw contains null values."
)

assert retrieval_queries["fold"].isin(
    [0, 1, 2, 3, 4]
).all(), (
    "Unexpected fold value detected."
)


# ------------------------------------------------------------
# 7. Population check
# ------------------------------------------------------------

assert len(retrieval_queries) == len(responses), (
    "Retrieval query population changed."
)

assert len(retrieval_queries) == 35_072, (
    "Unexpected retrieval query population."
)


# ------------------------------------------------------------
# 8. Session coverage
# ------------------------------------------------------------

query_sessions = set(
    retrieval_queries["session_id"]
)

session_table_ids = set(
    sessions["session_id"]
)

turn_session_ids = set(
    turns["session_id"]
)

missing_from_sessions = (
    query_sessions
    - session_table_ids
)

missing_from_turns = (
    query_sessions
    - turn_session_ids
)

assert missing_from_sessions == set(), (
    "Some query sessions are absent from sessions.parquet."
)

assert missing_from_turns == set(), (
    "Some query sessions are absent from turns.parquet."
)


# ------------------------------------------------------------
# 9. Objective coverage
# ------------------------------------------------------------

objective_ids = set(
    objectives["objective_uid"]
)

missing_objectives = (
    set(retrieval_queries["objective_uid"])
    - objective_ids
)

assert missing_objectives == set(), (
    "Some query objectives are absent from objectives.parquet."
)


# ------------------------------------------------------------
# 10. Objective text consistency
# ------------------------------------------------------------

objective_map = (
    objectives[
        [
            "objective_uid",
            "objective_raw",
        ]
    ]
    .drop_duplicates(
        "objective_uid"
    )
    .set_index(
        "objective_uid"
    )["objective_raw"]
)

mapped_objective_raw = (
    retrieval_queries["objective_uid"]
    .map(objective_map)
)

objective_mismatch = (
    mapped_objective_raw
    != retrieval_queries["objective_raw"]
).sum()

assert objective_mismatch == 0, (
    f"Objective identity mismatch: "
    f"{objective_mismatch}"
)


# ------------------------------------------------------------
# 11. Write artifact
# ------------------------------------------------------------

retrieval_queries_path = (
    R0_ROOT
    / "retrieval_queries.parquet"
)

retrieval_queries.to_parquet(
    retrieval_queries_path,
    index=False,
    compression="zstd",
)


# ------------------------------------------------------------
# 12. Reload verification
# ------------------------------------------------------------

reloaded_queries = pd.read_parquet(
    retrieval_queries_path
)

assert len(reloaded_queries) == 35_072

assert list(
    reloaded_queries.columns
) == QUERY_COLUMNS

assert reloaded_queries["response_id"].is_unique

assert not (
    set(reloaded_queries.columns)
    &
    PROHIBITED_QUERY_COLUMNS
)


# ------------------------------------------------------------
# 13. SHA256
# ------------------------------------------------------------

def sha256_file_r0(
    path,
    chunk_size=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


retrieval_queries_sha256 = (
    sha256_file_r0(
        retrieval_queries_path
    )
)


# ------------------------------------------------------------
# 14. Final status
# ------------------------------------------------------------

R0_RETRIEVAL_QUERIES_READY = True


print("\n" + "=" * 80)
print("R0 CELL 6 STATUS")
print("=" * 80)

print(
    f"Rows                         : "
    f"{len(retrieval_queries)}/35072"
)

print(
    f"Columns                      : "
    f"{len(retrieval_queries.columns)}/5"
)

print(
    f"response_id unique           : "
    f"{retrieval_queries['response_id'].is_unique}"
)

print(
    f"Session coverage             : "
    f"{len(missing_from_sessions) == 0}"
)

print(
    f"Turn coverage                : "
    f"{len(missing_from_turns) == 0}"
)

print(
    f"Objective coverage           : "
    f"{len(missing_objectives) == 0}"
)

print(
    f"Objective identity           : "
    f"{objective_mismatch == 0}"
)

print(
    f"Target leakage               : "
    f"{len(query_leakage)}"
)

print(
    f"Serialization verified       : "
    f"{len(reloaded_queries) == 35072}"
)

print(
    f"SHA256                       : "
    f"{retrieval_queries_sha256}"
)

print(
    f"R0_RETRIEVAL_QUERIES_READY   : "
    f"{R0_RETRIEVAL_QUERIES_READY}"
)

display(
    retrieval_queries.head(10)
)

print("=" * 80)

del reloaded_queries
gc.collect()

TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
CELL 6 — BUILD RETRIEVAL QUERIES

R0 CELL 6 STATUS
Rows                         : 35072/35072
Columns                      : 5/5
response_id unique           : True
Session coverage             : True
Turn coverage                : True
Objective coverage           : True
Objective identity           : True
Target leakage               : 0
Serialization verified       : True
SHA256                       : 59dc5626eb5d2ed776f1dbb76757781df80ea6a88c903f8b6f2d19484ca9c7c1
R0_RETRIEVAL_QUERIES_READY   : True


,response_id,session_id,objective_raw,objective_uid,fold
0,aaaavsh,bcaufvc,Knowing the value of each digit in numbers wit...,OBJ_42e7a829abfa2032e7f1e94885bcf1dc83649e8b55...,3
1,aaabhzi,eyutanf,Adding and subtracting tens to a 2-digit number.,OBJ_62a540f8645b1b14b4e0f078d1530b9fb0f8bf9493...,2
2,aaahpnz,juptkxd,Comparing and ordering fractions by finding a ...,OBJ_f33453a0ac95ec9160140d6bf47f02a59604f583ca...,4
3,aaajpom,ntwkcfj,Comparing fractions using reasoning.,OBJ_80630a8e6600e37ad3ec0a8f29ede170aa53a23c2e...,1
4,aaamwux,jqriibm,Counting in multiples.,OBJ_1beac46055fd137a49ac2a7960dd578fa0930ed674...,4
5,aaaqjzc,iykbtex,Seeing fractions as numbers.,OBJ_1b83907434d356ccff35f0745ca75ffab82d251ff5...,2
6,aabgafx,nbsxtoc,"Comparing acute, obtuse and reflex angles.",OBJ_2dbe25aac7926cdfcddc0506e0cc082a0b0e408ac6...,1
7,aabieaa,jetyiwh,Multiplying using short multiplication.,OBJ_90ae5176f8ead7d27dbd4bf8d92537fa68d457a716...,3
8,aabjsyx,ivrevge,"Ordering numbers up to 1,000,000.",OBJ_2c924edca01eae31127ddb801e9b9a84d4dc7abb03...,4
9,aabmjvr,arhtxbe,Using fact families to multiply and divide.,OBJ_bd3d36630462fbcf220e9b095a32647a64553c3946...,0


0

In [9]:
# ============================================================
# TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
# CELL 7 — BUILD OBJECTIVE CATALOGUE
# ============================================================

import gc
import hashlib
import os
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER")
print("CELL 7 — BUILD OBJECTIVE CATALOGUE")
print("=" * 80)


# ------------------------------------------------------------
# 0. Preconditions
# ------------------------------------------------------------

assert R0_CANONICAL_DISCOVERY_READY is True, (
    "Canonical artifacts were not discovered successfully."
)

assert R0_RELATIONAL_CONTRACT_READY is True, (
    "R0 relational contract is not ready."
)

assert R0_TURN_CONTRACT_READY is True, (
    "R0 turn contract is not ready."
)

assert R0_RETRIEVAL_QUERIES_READY is True, (
    "R0 retrieval queries are not ready."
)

assert R0_ROOT.exists(), (
    "R0 output root does not exist."
)

# ------------------------------------------------------------
# 1. Output path
# ------------------------------------------------------------

R0_OBJECTIVE_CATALOGUE_FINAL = (
    R0_ROOT / "objective_catalogue.parquet"
)

R0_OBJECTIVE_CATALOGUE_TMP = (
    R0_ROOT / ".objective_catalogue.parquet.tmp"
)

print("\nOutput root:")
print(R0_ROOT)

print("\nObjective catalogue:")
print(R0_OBJECTIVE_CATALOGUE_FINAL)


# ------------------------------------------------------------
# 2. Remove stale temporary artifact only
# ------------------------------------------------------------

if R0_OBJECTIVE_CATALOGUE_TMP.exists():
    R0_OBJECTIVE_CATALOGUE_TMP.unlink()
    print("\nRemoved stale temporary objective catalogue.")


# ------------------------------------------------------------
# 3. Exact retrieval-approved schema
#
# Keep this artifact intentionally minimal.
#
# Do NOT carry:
#   target
#   fold
#   response_count
#   session_count
#
# The catalogue is an identity/text lookup table only.
# ------------------------------------------------------------

R0_OBJECTIVE_CATALOGUE_COLUMNS = [
    "objective_uid",
    "objective_raw",
]


# ------------------------------------------------------------
# 4. Build catalogue from canonical objectives
# ------------------------------------------------------------

objective_catalogue = (
    objectives[
        R0_OBJECTIVE_CATALOGUE_COLUMNS
    ]
    .copy()
    .drop_duplicates()
    .sort_values(
        "objective_uid",
        kind="mergesort",
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Pre-write contract
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("OBJECTIVE CATALOGUE PRE-WRITE AUDIT")
print("=" * 80)

print(
    f"Rows      : {len(objective_catalogue)}"
)

print(
    f"Columns   : {len(objective_catalogue.columns)}"
)

print(
    f"Expected  : 398 rows / "
    f"{len(R0_OBJECTIVE_CATALOGUE_COLUMNS)} columns"
)

print(
    f"Columns   : {list(objective_catalogue.columns)}"
)


assert len(objective_catalogue) == 398, (
    "Objective catalogue population changed."
)

assert list(
    objective_catalogue.columns
) == R0_OBJECTIVE_CATALOGUE_COLUMNS, (
    "Objective catalogue schema mismatch."
)


# ------------------------------------------------------------
# 6. Identity checks
# ------------------------------------------------------------

objective_uid_unique = (
    objective_catalogue["objective_uid"].is_unique
)

objective_raw_nonnull = (
    objective_catalogue["objective_raw"].notna().all()
)

objective_raw_nonblank = (
    objective_catalogue["objective_raw"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
)

objective_uid_nonnull = (
    objective_catalogue["objective_uid"].notna().all()
)

print("\n" + "=" * 80)
print("OBJECTIVE IDENTITY AUDIT")
print("=" * 80)

print(
    f"objective_uid unique    : "
    f"{objective_uid_unique}"
)

print(
    f"objective_uid non-null  : "
    f"{objective_uid_nonnull}"
)

print(
    f"objective_raw non-null  : "
    f"{objective_raw_nonnull}"
)

print(
    f"objective_raw non-blank : "
    f"{objective_raw_nonblank}"
)

assert objective_uid_unique, (
    "objective_uid is not globally unique."
)

assert objective_uid_nonnull, (
    "objective_uid contains null values."
)

assert objective_raw_nonnull, (
    "objective_raw contains null values."
)

assert objective_raw_nonblank, (
    "objective_raw contains blank values."
)


# ------------------------------------------------------------
# 7. Objective identity consistency
#
# Every retrieval query objective_uid must resolve to exactly
# one objective catalogue row.
# ------------------------------------------------------------

query_objective_ids = set(
    retrieval_queries["objective_uid"]
)

catalogue_objective_ids = set(
    objective_catalogue["objective_uid"]
)

missing_objectives = (
    query_objective_ids
    -
    catalogue_objective_ids
)

unexpected_catalogue_objectives = (
    catalogue_objective_ids
    -
    query_objective_ids
)

print("\n" + "=" * 80)
print("QUERY → OBJECTIVE CATALOGUE COVERAGE")
print("=" * 80)

print(
    f"Query objective IDs      : "
    f"{len(query_objective_ids)}"
)

print(
    f"Catalogue objective IDs  : "
    f"{len(catalogue_objective_ids)}"
)

print(
    f"Missing catalogue IDs    : "
    f"{len(missing_objectives)}"
)

print(
    f"Unused catalogue IDs     : "
    f"{len(unexpected_catalogue_objectives)}"
)

assert len(missing_objectives) == 0, (
    "Retrieval queries reference unknown objective_uid."
)

assert len(unexpected_catalogue_objectives) == 0, (
    "Objective catalogue contains objectives absent from "
    "retrieval query population."
)


# ------------------------------------------------------------
# 8. Target / evaluation isolation
# ------------------------------------------------------------

PROHIBITED_R0_OBJECTIVE_COLUMNS = {
    "target",
    "label",
    "prediction",
    "positive_rate",
    "target_mean",
    "target_count",
    "positive_count",
    "negative_count",
    "fold",
    "response_count",
    "session_count",
}

prohibited_objective_columns = sorted(
    {
        str(col).lower()
        for col in objective_catalogue.columns
    }
    &
    PROHIBITED_R0_OBJECTIVE_COLUMNS
)

print("\n" + "=" * 80)
print("TARGET / EVALUATION ISOLATION")
print("=" * 80)

print(
    f"Prohibited fields : "
    f"{prohibited_objective_columns}"
)

assert prohibited_objective_columns == [], (
    "Target/evaluation-derived metadata leaked into "
    "objective catalogue."
)


# ------------------------------------------------------------
# 9. Write temporary parquet
# ------------------------------------------------------------

objective_table = pa.Table.from_pandas(
    objective_catalogue,
    preserve_index=False,
)

pq.write_table(
    objective_table,
    R0_OBJECTIVE_CATALOGUE_TMP,
    compression="zstd",
    use_dictionary=True,
)

del objective_table
gc.collect()


# ------------------------------------------------------------
# 10. Reload / exact retrieval schema verification
# ------------------------------------------------------------

objective_metadata = pq.read_metadata(
    R0_OBJECTIVE_CATALOGUE_TMP
)

objective_schema = pq.read_schema(
    R0_OBJECTIVE_CATALOGUE_TMP
)

objective_rows = int(
    objective_metadata.num_rows
)

expected_objective_schema = pa.schema([
    pa.field("objective_uid", pa.string()),
    pa.field("objective_raw", pa.string()),
])

objective_schema_types_match = (
    objective_schema.names == expected_objective_schema.names
    and objective_schema.types == expected_objective_schema.types
)

print("\n" + "=" * 80)
print("SERIALIZATION AUDIT")
print("=" * 80)

print(f"Reloaded rows   : {objective_rows}")
print(f"Reloaded fields : {len(objective_schema)}")
print(f"Schema types match : {objective_schema_types_match}")

assert objective_rows == 398
assert objective_schema_types_match


# ------------------------------------------------------------
# 11. SHA256 helper
# ------------------------------------------------------------

def sha256_file_r0_objective(
    path,
    chunk_size=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


objective_tmp_sha256 = sha256_file_r0_objective(
    R0_OBJECTIVE_CATALOGUE_TMP
)

print(
    f"\nTemporary SHA256:\n"
    f"{objective_tmp_sha256}"
)


# ------------------------------------------------------------
# 11. SHA256
# ------------------------------------------------------------

objective_tmp_sha256 = sha256_file_r0_objective(
    R0_OBJECTIVE_CATALOGUE_TMP
)

print(
    f"\nTemporary SHA256:\n"
    f"{objective_tmp_sha256}"
)


# ------------------------------------------------------------
# 12. Release Python references
# ------------------------------------------------------------

del objective_metadata
del objective_schema
gc.collect()


# ------------------------------------------------------------
# 13. Atomic promotion — ONLY ONCE
# ------------------------------------------------------------

assert R0_OBJECTIVE_CATALOGUE_TMP.exists(), (
    "Temporary objective catalogue disappeared before promotion."
)

if R0_OBJECTIVE_CATALOGUE_FINAL.exists():
    R0_OBJECTIVE_CATALOGUE_FINAL.unlink()

os.replace(
    R0_OBJECTIVE_CATALOGUE_TMP,
    R0_OBJECTIVE_CATALOGUE_FINAL,
)

assert R0_OBJECTIVE_CATALOGUE_FINAL.exists()
assert not R0_OBJECTIVE_CATALOGUE_TMP.exists()


# ------------------------------------------------------------
# 14. Final verification
# ------------------------------------------------------------

final_metadata = pq.read_metadata(
    R0_OBJECTIVE_CATALOGUE_FINAL
)

final_schema = pq.read_schema(
    R0_OBJECTIVE_CATALOGUE_FINAL
)

final_objective_rows = int(
    final_metadata.num_rows
)

final_objective_schema_types_match = (
    final_schema.names == expected_objective_schema.names
    and final_schema.types == expected_objective_schema.types
)

final_objective_sha256 = sha256_file_r0_objective(
    R0_OBJECTIVE_CATALOGUE_FINAL
)

print("\n" + "=" * 80)
print("FINAL OBJECTIVE CATALOGUE VERIFICATION")
print("=" * 80)

print(f"Rows          : {final_objective_rows}")
print(f"Fields        : {len(final_schema)}")
print(f"Schema stable : {final_objective_schema_types_match}")
print(f"SHA256        : {final_objective_sha256}")

assert final_objective_rows == 398
assert final_objective_schema_types_match


# ------------------------------------------------------------
# 15. Final gate
# ------------------------------------------------------------

R0_OBJECTIVE_CATALOGUE_READY = all([
    R0_OBJECTIVE_CATALOGUE_FINAL.exists(),
    final_objective_rows == 398,
    final_objective_schema_types_match,
    objective_uid_unique,
    objective_uid_nonnull,
    objective_raw_nonnull,
    objective_raw_nonblank,
    len(missing_objectives) == 0,
    len(unexpected_catalogue_objectives) == 0,
    prohibited_objective_columns == [],
])

print("\n" + "=" * 80)
print("R0 CELL 7 STATUS")
print("=" * 80)

print(
    f"Objective rows             : "
    f"{final_objective_rows}/398"
)

print(
    f"Objective UID unique       : "
    f"{objective_uid_unique}"
)

print(
    f"Query → objective coverage : "
    f"{len(missing_objectives) == 0}"
)

print(
    f"Unused objectives          : "
    f"{len(unexpected_catalogue_objectives)}"
)

print(
    f"Target/evaluation leakage  : "
    f"{len(prohibited_objective_columns)}"
)

print(
    f"Serialization verified     : "
    f"{final_objective_schema_types_match}"
)

print(
    f"SHA256                     : "
    f"{final_objective_sha256}"
)

print(
    f"R0_OBJECTIVE_CATALOGUE_READY : "
    f"{R0_OBJECTIVE_CATALOGUE_READY}"
)

assert R0_OBJECTIVE_CATALOGUE_READY

# ------------------------------------------------------------
# 16. Release memory
# ------------------------------------------------------------

import gc

for _name in [
    "objective_catalogue",
    "objective_pf",
    "final_objective_pf",
    "catalogue_reload_sample",
    "catalogue_source_sample",
    "objective_metadata",
    "objective_schema",
    "final_metadata",
    "final_schema",
]:
    globals().pop(_name, None)

gc.collect()

print("R0 Cell 7 memory cleanup: PASS")

TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
CELL 7 — BUILD OBJECTIVE CATALOGUE

Output root:
D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input

Objective catalogue:
D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\objective_catalogue.parquet

OBJECTIVE CATALOGUE PRE-WRITE AUDIT
Rows      : 398
Columns   : 2
Expected  : 398 rows / 2 columns
Columns   : ['objective_uid', 'objective_raw']

OBJECTIVE IDENTITY AUDIT
objective_uid unique    : True
objective_uid non-null  : True
objective_raw non-null  : True
objective_raw non-blank : True

QUERY → OBJECTIVE CATALOGUE COVERAGE
Query objective IDs      : 398
Catalogue objective IDs  : 398
Missing catalogue IDs    : 0
Unused catalogue IDs     : 0

TARGET / EVALUATION ISOLATION
Prohibited fields : []

SERIALIZATION AUDIT
Reloaded rows   : 398
Reloaded fields : 2
Schema types match : True

Temporary SHA256:
945a59ac004fa3be0818512db3d974516dec6b51ba66e5ad34c37fe67772fee2

Temporary S

In [11]:
# ============================================================
# TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
# CELL 8 — BUILD SESSION TURN INDEX
# ============================================================

import gc
import hashlib
import os
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER")
print("CELL 8 — BUILD SESSION TURN INDEX")
print("=" * 80)


# ------------------------------------------------------------
# 0. Preconditions
# ------------------------------------------------------------

assert R0_CANONICAL_DISCOVERY_READY is True, (
    "Canonical artifacts were not discovered successfully."
)

assert R0_RELATIONAL_CONTRACT_READY is True, (
    "R0 relational contract is not ready."
)

assert R0_TURN_CONTRACT_READY is True, (
    "R0 turn contract is not ready."
)

assert R0_RETRIEVAL_QUERIES_READY is True, (
    "R0 retrieval queries are not ready."
)

assert R0_OBJECTIVE_CATALOGUE_READY is True, (
    "R0 objective catalogue is not ready."
)

assert R0_ROOT.exists(), (
    "R0 output root does not exist."
)


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

R0_SESSION_TURN_INDEX_FINAL = (
    R0_ROOT / "session_turn_index.parquet"
)

R0_SESSION_TURN_INDEX_TMP = (
    R0_ROOT / ".session_turn_index.parquet.tmp"
)

TURNS_CANONICAL_PATH = canonical_paths["turns"]

print("\nCanonical turns:")
print(TURNS_CANONICAL_PATH)

print("\nSession-turn index:")
print(R0_SESSION_TURN_INDEX_FINAL)


# ------------------------------------------------------------
# 2. Exact retrieval-approved schema
#
# IMPORTANT:
# This is the contract already certified by Cell 5.
#
# Do NOT add:
#   target
#   fold
#   response_id
#   objective_uid
#   objective_raw
#   source provenance fields
#   target-derived statistics
# ------------------------------------------------------------

R0_SESSION_TURN_INDEX_COLUMNS = [
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "text_norm",
    "relative_turn_position",
    "previous_role",
    "next_role",
    "speaker_switch",
    "time_since_previous_turn",
    "elapsed_from_session_start",
    "ordering_method",
    "ordering_confidence",
    "ordering_comparability",
]


# ------------------------------------------------------------
# 3. Remove stale temporary artifact
# ------------------------------------------------------------

if R0_SESSION_TURN_INDEX_TMP.exists():
    try:
        R0_SESSION_TURN_INDEX_TMP.unlink()
        print("\nRemoved stale temporary session-turn index.")
    except PermissionError as e:
        raise RuntimeError(
            "Temporary session-turn index is locked. "
            "Restart the kernel before rerunning Cell 8."
        ) from e


# ------------------------------------------------------------
# 4. Canonical source inspection
# ------------------------------------------------------------

source_pf = pq.ParquetFile(
    TURNS_CANONICAL_PATH
)

source_schema = source_pf.schema_arrow
source_rows = int(
    source_pf.metadata.num_rows
)

source_columns = set(
    source_schema.names
)

missing_source_columns = sorted(
    set(R0_SESSION_TURN_INDEX_COLUMNS)
    - source_columns
)

assert missing_source_columns == [], (
    "Canonical turns are missing required R0 fields: "
    f"{missing_source_columns}"
)

print("\n" + "=" * 80)
print("SOURCE CONTRACT")
print("=" * 80)

print(
    f"Canonical turn rows : "
    f"{source_rows:,}"
)

print(
    f"Required R0 fields  : "
    f"{len(R0_SESSION_TURN_INDEX_COLUMNS)}"
)

print(
    f"Missing fields      : "
    f"{missing_source_columns}"
)

assert source_rows == 6_139_854, (
    "Canonical turn population changed."
)


# ------------------------------------------------------------
# 5. Build exact Arrow schema from canonical source
#
# Preserve canonical Arrow types.
# Do not infer through pandas.
# ------------------------------------------------------------

R0_SESSION_TURN_INDEX_SCHEMA = pa.schema([
    source_schema.field(name)
    for name in R0_SESSION_TURN_INDEX_COLUMNS
])

print("\n" + "=" * 80)
print("R0 SESSION-TURN INDEX SCHEMA")
print("=" * 80)

print(R0_SESSION_TURN_INDEX_SCHEMA)


# ------------------------------------------------------------
# 6. Explicit target / evaluation isolation
# ------------------------------------------------------------

PROHIBITED_R0_TURN_FIELDS = {
    "target",
    "label",
    "prediction",
    "fold",
    "objective_uid",
    "objective_raw",
    "response_id",
    "response_count",
    "session_count",
}

prohibited_fields = sorted(
    {
        str(col).lower()
        for col in R0_SESSION_TURN_INDEX_COLUMNS
    }
    &
    PROHIBITED_R0_TURN_FIELDS
)

print("\n" + "=" * 80)
print("TARGET / EVALUATION ISOLATION")
print("=" * 80)

print(
    f"Prohibited fields : "
    f"{prohibited_fields}"
)

assert prohibited_fields == [], (
    "Prohibited target/evaluation fields entered "
    "the session-turn retrieval index."
)


# ------------------------------------------------------------
# 7. Streaming write
#
# Canonical turns are already certified as session-contiguous
# and turn_index ordered by Cell 5.
#
# Therefore:
#   NO global sort
#   NO pandas concat
#   NO 6M-row duplicate DataFrame
#
# We preserve canonical order exactly.
# ------------------------------------------------------------

writer = None
rows_written = 0
batches_written = 0

try:

    for batch in source_pf.iter_batches(
        batch_size=100_000,
        columns=R0_SESSION_TURN_INDEX_COLUMNS,
    ):

        batch_table = pa.Table.from_batches(
            [batch],
            schema=R0_SESSION_TURN_INDEX_SCHEMA,
        )

        if writer is None:
            writer = pq.ParquetWriter(
                R0_SESSION_TURN_INDEX_TMP,
                R0_SESSION_TURN_INDEX_SCHEMA,
                compression="zstd",
                use_dictionary=True,
            )

        writer.write_table(
            batch_table,
            row_group_size=batch.num_rows,
        )

        rows_written += int(batch.num_rows)
        batches_written += 1

        if (
            batches_written % 10 == 0
            or
            rows_written == source_rows
        ):
            print(
                f"Written batches : "
                f"{batches_written:,} | "
                f"rows : {rows_written:,}/"
                f"{source_rows:,}"
            )

finally:

    if writer is not None:
        writer.close()

    # Explicitly release source reader before promotion.
    del source_pf
    gc.collect()


# ------------------------------------------------------------
# 8. Write accounting
# ------------------------------------------------------------

assert R0_SESSION_TURN_INDEX_TMP.exists(), (
    "Temporary session-turn index was not created."
)

assert rows_written == source_rows, (
    f"Session-turn index row mismatch: "
    f"{rows_written} != {source_rows}"
)

print("\n" + "=" * 80)
print("WRITE ACCOUNTING")
print("=" * 80)

print(
    f"Rows written : "
    f"{rows_written:,}/{source_rows:,}"
)

print(
    f"Batches      : "
    f"{batches_written:,}"
)


# ------------------------------------------------------------
# 9. Serialized metadata/schema verification
#
# Use metadata/schema readers rather than ParquetFile so that
# no long-lived reader remains open during promotion.
# ------------------------------------------------------------

tmp_metadata = pq.read_metadata(
    R0_SESSION_TURN_INDEX_TMP
)

tmp_schema = pq.read_schema(
    R0_SESSION_TURN_INDEX_TMP
)

tmp_rows = int(
    tmp_metadata.num_rows
)

tmp_schema_types_match = (
    tmp_schema.names
    == R0_SESSION_TURN_INDEX_SCHEMA.names
    and
    tmp_schema.types
    == R0_SESSION_TURN_INDEX_SCHEMA.types
)

print("\n" + "=" * 80)
print("SERIALIZATION AUDIT")
print("=" * 80)

print(
    f"Reloaded rows       : "
    f"{tmp_rows:,}"
)

print(
    f"Reloaded fields     : "
    f"{len(tmp_schema)}"
)

print(
    f"Schema types match  : "
    f"{tmp_schema_types_match}"
)

assert tmp_rows == 6_139_854, (
    "Serialized session-turn index row count changed."
)

assert tmp_schema_types_match, (
    "Serialized session-turn index field/type schema differs "
    "from the certified R0 schema."
)


# ------------------------------------------------------------
# 10. SHA256 helper
# ------------------------------------------------------------

def sha256_file_r0_session_turn_index(
    path,
    chunk_size=8 * 1024 * 1024,
):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


session_turn_index_tmp_sha256 = (
    sha256_file_r0_session_turn_index(
        R0_SESSION_TURN_INDEX_TMP
    )
)

print("\nTemporary SHA256:")
print(session_turn_index_tmp_sha256)


# ------------------------------------------------------------
# 11. Release metadata objects before promotion
# ------------------------------------------------------------

del tmp_metadata
del tmp_schema

gc.collect()


# ------------------------------------------------------------
# 12. Atomic promotion — exactly once
# ------------------------------------------------------------

if R0_SESSION_TURN_INDEX_FINAL.exists():
    try:
        R0_SESSION_TURN_INDEX_FINAL.unlink()
    except PermissionError as e:
        raise RuntimeError(
            "Final session-turn index is locked. "
            "Restart the kernel before rerunning Cell 8."
        ) from e

try:

    os.replace(
        R0_SESSION_TURN_INDEX_TMP,
        R0_SESSION_TURN_INDEX_FINAL,
    )

except PermissionError as e:

    raise RuntimeError(
        "Windows refused atomic promotion because the temporary "
        "session-turn index is still locked. Restart the kernel "
        "and rerun Cell 8."
    ) from e


assert R0_SESSION_TURN_INDEX_FINAL.exists(), (
    "Final session-turn index was not published."
)

assert not R0_SESSION_TURN_INDEX_TMP.exists(), (
    "Temporary session-turn index still exists after promotion."
)


# ------------------------------------------------------------
# 13. Final artifact verification
# ------------------------------------------------------------

final_metadata = pq.read_metadata(
    R0_SESSION_TURN_INDEX_FINAL
)

final_schema = pq.read_schema(
    R0_SESSION_TURN_INDEX_FINAL
)

final_rows = int(
    final_metadata.num_rows
)

final_schema_types_match = (
    final_schema.names
    == R0_SESSION_TURN_INDEX_SCHEMA.names
    and
    final_schema.types
    == R0_SESSION_TURN_INDEX_SCHEMA.types
)

final_sha256 = (
    sha256_file_r0_session_turn_index(
        R0_SESSION_TURN_INDEX_FINAL
    )
)

print("\n" + "=" * 80)
print("FINAL SESSION-TURN INDEX VERIFICATION")
print("=" * 80)

print(
    f"Rows          : "
    f"{final_rows:,}/{source_rows:,}"
)

print(
    f"Fields        : "
    f"{len(final_schema)}/{len(R0_SESSION_TURN_INDEX_COLUMNS)}"
)

print(
    f"Schema stable : "
    f"{final_schema_types_match}"
)

print(
    f"SHA256        : "
    f"{final_sha256}"
)

assert final_rows == source_rows, (
    "Final session-turn index row count changed."
)

assert final_schema_types_match, (
    "Final session-turn index schema changed."
)


# ------------------------------------------------------------
# 14. Retrieval coverage verification
#
# Every labelled response session must exist in the index.
# We use the already validated retrieval query table.
# ------------------------------------------------------------

query_sessions = set(
    retrieval_queries["session_id"]
)

# Read only session_id for coverage.
final_session_ids = set()

coverage_pf = pq.ParquetFile(
    R0_SESSION_TURN_INDEX_FINAL
)

try:

    for batch in coverage_pf.iter_batches(
        batch_size=250_000,
        columns=["session_id"],
    ):

        final_session_ids.update(
            batch.column("session_id").to_pylist()
        )

finally:

    del coverage_pf
    gc.collect()

missing_response_sessions = (
    query_sessions
    -
    final_session_ids
)

unexpected_sessions = (
    final_session_ids
    -
    set(
        sessions["session_id"]
    )
)

print("\n" + "=" * 80)
print("SESSION COVERAGE AUDIT")
print("=" * 80)

print(
    f"Query sessions       : "
    f"{len(query_sessions):,}"
)

print(
    f"Indexed sessions     : "
    f"{len(final_session_ids):,}"
)

print(
    f"Missing query sessions : "
    f"{len(missing_response_sessions):,}"
)

print(
    f"Unknown indexed sessions : "
    f"{len(unexpected_sessions):,}"
)

assert len(missing_response_sessions) == 0, (
    "Some retrieval query sessions have no session-turn index."
)

assert len(unexpected_sessions) == 0, (
    "Session-turn index contains sessions absent from "
    "canonical sessions."
)


# ------------------------------------------------------------
# 15. Final gate
# ------------------------------------------------------------

R0_SESSION_TURN_INDEX_READY = all([
    R0_SESSION_TURN_INDEX_FINAL.exists(),
    final_rows == 6_139_854,
    len(final_schema) == 14,
    final_schema_types_match,
    prohibited_fields == [],
    len(missing_response_sessions) == 0,
    len(unexpected_sessions) == 0,
    not R0_SESSION_TURN_INDEX_TMP.exists(),
])

print("\n" + "=" * 80)
print("R0 CELL 8 STATUS")
print("=" * 80)

print(
    f"Turn rows                 : "
    f"{final_rows:,}/6,139,854"
)

print(
    f"Schema fields             : "
    f"{len(final_schema)}/14"
)

print(
    f"Response-session coverage : "
    f"{len(missing_response_sessions) == 0}"
)

print(
    f"Unknown sessions          : "
    f"{len(unexpected_sessions)}"
)

print(
    f"Target/evaluation leakage : "
    f"{len(prohibited_fields)}"
)

print(
    f"Serialization verified    : "
    f"{final_schema_types_match}"
)

print(
    f"SHA256                    : "
    f"{final_sha256}"
)

print(
    f"R0_SESSION_TURN_INDEX_READY : "
    f"{R0_SESSION_TURN_INDEX_READY}"
)

print("=" * 80)

assert R0_SESSION_TURN_INDEX_READY

TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
CELL 8 — BUILD SESSION TURN INDEX

Canonical turns:
D:\Competition\Trace-the-race-local\scratch_mastery_outputs\01_data_foundation\03_integrity\canonical\turns.parquet

Session-turn index:
D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\session_turn_index.parquet

SOURCE CONTRACT
Canonical turn rows : 6,139,854
Required R0 fields  : 14
Missing fields      : []

R0 SESSION-TURN INDEX SCHEMA
session_id: string not null
turn_uid: string not null
turn_index: int32 not null
role: string not null
text_norm: large_string not null
relative_turn_position: float not null
previous_role: string
next_role: string
speaker_switch: bool not null
time_since_previous_turn: double
elapsed_from_session_start: double
ordering_method: string not null
ordering_confidence: string not null
ordering_comparability: string not null

TARGET / EVALUATION ISOLATION
Prohibited fields : []
Written batches : 10 | rows : 1,000,000/6,139,854
Wri

In [15]:
# ============================================================
# TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
# CELL 9 — FINAL R0 AUDIT + MANIFEST + FREEZE GATE
# ============================================================

import gc
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


print("=" * 80)
print("TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER")
print("CELL 9 — FINAL R0 AUDIT + MANIFEST + FREEZE GATE")
print("=" * 80)


# ============================================================
# 0. PRECONDITIONS
# ============================================================

assert R0_CANONICAL_DISCOVERY_READY is True, (
    "Canonical artifact discovery is not ready."
)

assert R0_ROOT.exists(), (
    "R0 output root does not exist."
)

assert R0_ROOT.is_dir(), (
    "R0 output root is not a directory."
)


# ============================================================
# 1. EXPECTED ARTIFACT PATHS
# ============================================================

R0_RETRIEVAL_QUERIES_FINAL = (
    R0_ROOT / "retrieval_queries.parquet"
)

R0_SESSION_TURN_INDEX_FINAL = (
    R0_ROOT / "session_turn_index.parquet"
)

R0_OBJECTIVE_CATALOGUE_FINAL = (
    R0_ROOT / "objective_catalogue.parquet"
)

R0_MANIFEST_FINAL = (
    R0_ROOT / "r0_manifest.json"
)


print("\n" + "=" * 80)
print("R0 ARTIFACT PATHS")
print("=" * 80)

print(f"retrieval_queries     : {R0_RETRIEVAL_QUERIES_FINAL}")
print(f"session_turn_index    : {R0_SESSION_TURN_INDEX_FINAL}")
print(f"objective_catalogue   : {R0_OBJECTIVE_CATALOGUE_FINAL}")
print(f"manifest              : {R0_MANIFEST_FINAL}")


# ============================================================
# 2. ARTIFACT EXISTENCE CHECK
# ============================================================

artifact_paths = {
    "retrieval_queries": R0_RETRIEVAL_QUERIES_FINAL,
    "session_turn_index": R0_SESSION_TURN_INDEX_FINAL,
    "objective_catalogue": R0_OBJECTIVE_CATALOGUE_FINAL,
}

artifact_exists = {
    name: path.exists()
    for name, path in artifact_paths.items()
}

print("\n" + "=" * 80)
print("ARTIFACT EXISTENCE")
print("=" * 80)

for name, exists in artifact_exists.items():
    print(f"{name:24s}: {exists}")

assert all(artifact_exists.values()), (
    "One or more required R0 artifacts are missing."
)


# ============================================================
# 3. LOAD FINAL ARTIFACTS FROM DISK
#
# Important:
# The freeze gate must validate the serialized artifacts,
# not merely the in-memory DataFrames.
# ============================================================

retrieval_queries_final = pd.read_parquet(
    R0_RETRIEVAL_QUERIES_FINAL
)

session_turn_index_final = pd.read_parquet(
    R0_SESSION_TURN_INDEX_FINAL
)

objective_catalogue_final = pd.read_parquet(
    R0_OBJECTIVE_CATALOGUE_FINAL
)


# ============================================================
# 4. POPULATION AUDIT
# ============================================================

print("\n" + "=" * 80)
print("R0 POPULATION AUDIT")
print("=" * 80)

print(
    f"Retrieval query rows : "
    f"{len(retrieval_queries_final):,}"
)

print(
    f"Expected              : "
    f"{len(responses):,}"
)

print(
    f"Session-turn rows     : "
    f"{len(session_turn_index_final):,}"
)

print(
    f"Expected              : "
    f"{len(turns):,}"
)

print(
    f"Objective rows        : "
    f"{len(objective_catalogue_final):,}"
)

print(
    f"Expected              : "
    f"{len(objectives):,}"
)

assert len(retrieval_queries_final) == len(responses), (
    "Retrieval query population changed."
)

assert len(session_turn_index_final) == len(turns), (
    "Session-turn index population changed."
)

assert len(objective_catalogue_final) == len(objectives), (
    "Objective catalogue population changed."
)


# ============================================================
# 5. EXACT SCHEMA CONTRACTS
# ============================================================

EXPECTED_QUERY_COLUMNS = [
    "response_id",
    "session_id",
    "objective_raw",
    "objective_uid",
    "fold",
]

EXPECTED_TURN_COLUMNS = [
    "session_id",
    "turn_uid",
    "turn_index",
    "role",
    "text_norm",
]

# Optional fields may be present after the five required fields.
EXPECTED_OBJECTIVE_COLUMNS = [
    "objective_uid",
    "objective_raw",
]


print("\n" + "=" * 80)
print("R0 SCHEMA AUDIT")
print("=" * 80)

print("\nRetrieval queries:")
print(list(retrieval_queries_final.columns))

print("\nSession-turn index:")
print(list(session_turn_index_final.columns))

print("\nObjective catalogue:")
print(list(objective_catalogue_final.columns))


# Retrieval queries must contain exactly the approved columns.
assert list(retrieval_queries_final.columns) == EXPECTED_QUERY_COLUMNS, (
    "Retrieval query schema mismatch."
)

# Session-turn index may contain approved optional structural fields,
# but the core fields must exist.
missing_turn_columns = [
    c for c in EXPECTED_TURN_COLUMNS
    if c not in session_turn_index_final.columns
]

assert missing_turn_columns == [], (
    f"Session-turn index missing required fields: "
    f"{missing_turn_columns}"
)

# Objective catalogue is intentionally minimal.
assert list(objective_catalogue_final.columns) == EXPECTED_OBJECTIVE_COLUMNS, (
    "Objective catalogue schema mismatch."
)

print("\nR0 schema contract: PASS")


# ============================================================
# 6. TARGET / EVALUATION ISOLATION
# ============================================================

PROHIBITED_QUERY_COLUMNS = {
    "target",
    "label",
    "prediction",
    "positive_rate",
    "target_mean",
    "target_count",
    "positive_count",
    "negative_count",
}

PROHIBITED_OBJECTIVE_COLUMNS = {
    "target",
    "label",
    "prediction",
    "positive_rate",
    "target_mean",
    "target_count",
    "positive_count",
    "negative_count",
    "response_count",
    "session_count",
}

query_prohibited = sorted(
    {
        str(c).lower()
        for c in retrieval_queries_final.columns
    }
    & PROHIBITED_QUERY_COLUMNS
)

objective_prohibited = sorted(
    {
        str(c).lower()
        for c in objective_catalogue_final.columns
    }
    & PROHIBITED_OBJECTIVE_COLUMNS
)

print("\n" + "=" * 80)
print("TARGET / EVALUATION ISOLATION")
print("=" * 80)

print(
    f"Retrieval query prohibited fields : "
    f"{query_prohibited}"
)

print(
    f"Objective prohibited fields       : "
    f"{objective_prohibited}"
)

assert query_prohibited == [], (
    "Target/evaluation-derived field leaked into retrieval queries."
)

assert objective_prohibited == [], (
    "Target/evaluation-derived field leaked into objective catalogue."
)

print("\nR0 target/evaluation isolation: PASS")


# ============================================================
# 7. RESPONSE IDENTITY AUDIT
# ============================================================

print("\n" + "=" * 80)
print("RETRIEVAL QUERY IDENTITY")
print("=" * 80)

query_response_unique = (
    retrieval_queries_final["response_id"].is_unique
)

query_response_nonnull = (
    retrieval_queries_final["response_id"].notna().all()
)

print(
    f"response_id unique   : "
    f"{query_response_unique}"
)

print(
    f"response_id non-null : "
    f"{query_response_nonnull}"
)

assert query_response_unique, (
    "response_id is not unique in retrieval queries."
)

assert query_response_nonnull, (
    "response_id contains null values."
)


# ============================================================
# 8. SESSION COVERAGE AUDIT
# ============================================================

query_sessions = set(
    retrieval_queries_final["session_id"]
)

turn_sessions = set(
    session_turn_index_final["session_id"]
)

missing_turn_sessions = (
    query_sessions - turn_sessions
)

extra_turn_sessions = (
    turn_sessions - query_sessions
)

print("\n" + "=" * 80)
print("QUERY → SESSION-TURN COVERAGE")
print("=" * 80)

print(
    f"Query sessions       : "
    f"{len(query_sessions):,}"
)

print(
    f"Turn-index sessions   : "
    f"{len(turn_sessions):,}"
)

print(
    f"Missing turn sessions : "
    f"{len(missing_turn_sessions):,}"
)

print(
    f"Extra turn sessions   : "
    f"{len(extra_turn_sessions):,}"
)

assert len(missing_turn_sessions) == 0, (
    "Some retrieval-query sessions have no turns."
)

assert len(extra_turn_sessions) == 0, (
    "Session-turn index contains sessions outside "
    "the labelled response population."
)

print("\nR0 session coverage: PASS")


# ============================================================
# 9. OBJECTIVE IDENTITY AUDIT
# ============================================================

query_objective_ids = set(
    retrieval_queries_final["objective_uid"]
)

catalogue_objective_ids = set(
    objective_catalogue_final["objective_uid"]
)

missing_objective_ids = (
    query_objective_ids - catalogue_objective_ids
)

extra_objective_ids = (
    catalogue_objective_ids - query_objective_ids
)

objective_uid_unique = (
    objective_catalogue_final["objective_uid"].is_unique
)

objective_uid_nonnull = (
    objective_catalogue_final["objective_uid"].notna().all()
)

objective_raw_nonnull = (
    objective_catalogue_final["objective_raw"].notna().all()
)

objective_raw_nonblank = (
    objective_catalogue_final["objective_raw"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
)

print("\n" + "=" * 80)
print("OBJECTIVE IDENTITY AUDIT")
print("=" * 80)

print(
    f"Query objective IDs      : "
    f"{len(query_objective_ids):,}"
)

print(
    f"Catalogue objective IDs  : "
    f"{len(catalogue_objective_ids):,}"
)

print(
    f"Missing catalogue IDs    : "
    f"{len(missing_objective_ids):,}"
)

print(
    f"Extra catalogue IDs      : "
    f"{len(extra_objective_ids):,}"
)

print(
    f"objective_uid unique     : "
    f"{objective_uid_unique}"
)

print(
    f"objective_uid non-null   : "
    f"{objective_uid_nonnull}"
)

print(
    f"objective_raw non-null   : "
    f"{objective_raw_nonnull}"
)

print(
    f"objective_raw non-blank  : "
    f"{objective_raw_nonblank}"
)

assert len(missing_objective_ids) == 0, (
    "Retrieval queries reference unknown objective_uid."
)

assert len(extra_objective_ids) == 0, (
    "Objective catalogue contains unused objective_uid."
)

assert objective_uid_unique, (
    "objective_uid is not unique."
)

assert objective_uid_nonnull, (
    "objective_uid contains null values."
)

assert objective_raw_nonnull, (
    "objective_raw contains null values."
)

assert objective_raw_nonblank, (
    "objective_raw contains blank values."
)

print("\nR0 objective coverage: PASS")


# ============================================================
# 10. TURN IDENTITY AUDIT
# ============================================================

turn_uid_unique = (
    session_turn_index_final["turn_uid"].is_unique
)

turn_uid_nonnull = (
    session_turn_index_final["turn_uid"].notna().all()
)

session_id_nonnull = (
    session_turn_index_final["session_id"].notna().all()
)

turn_index_nonnull = (
    session_turn_index_final["turn_index"].notna().all()
)

role_nonnull = (
    session_turn_index_final["role"].notna().all()
)

text_norm_nonnull = (
    session_turn_index_final["text_norm"].notna().all()
)

print("\n" + "=" * 80)
print("TURN IDENTITY AUDIT")
print("=" * 80)

print(f"turn_uid unique       : {turn_uid_unique}")
print(f"turn_uid non-null     : {turn_uid_nonnull}")
print(f"session_id non-null   : {session_id_nonnull}")
print(f"turn_index non-null   : {turn_index_nonnull}")
print(f"role non-null         : {role_nonnull}")
print(f"text_norm non-null    : {text_norm_nonnull}")

assert turn_uid_unique, (
    "turn_uid is not globally unique."
)

assert turn_uid_nonnull, (
    "turn_uid contains null values."
)

assert session_id_nonnull, (
    "session_id contains null values."
)

assert turn_index_nonnull, (
    "turn_index contains null values."
)

assert role_nonnull, (
    "role contains null values."
)

assert text_norm_nonnull, (
    "text_norm contains null values."
)

print("\nR0 turn identity: PASS")


# ============================================================
# 11. WITHIN-SESSION TURN ORDER AUDIT
# ============================================================

print("\n" + "=" * 80)
print("WITHIN-SESSION TURN ORDER AUDIT")
print("=" * 80)

order_failures = 0
census_failures = 0

for session_id, g in session_turn_index_final.groupby(
    "session_id",
    sort=False,
):
    idx = g["turn_index"].to_numpy()

    if len(idx) == 0:
        order_failures += 1
        continue

    if len(idx) != len(set(idx)):
        order_failures += 1
        continue

    if len(idx) > 1:
        if not (idx[1:] > idx[:-1]).all():
            order_failures += 1
            continue

    expected_idx = list(range(len(idx)))

    if idx.tolist() != expected_idx:
        census_failures += 1

print(
    f"Turn-index order failures : "
    f"{order_failures}"
)

print(
    f"Turn-index census failures : "
    f"{census_failures}"
)

assert order_failures == 0, (
    "Within-session turn ordering failure detected."
)

assert census_failures == 0, (
    "Within-session turn census failure detected."
)

print("\nR0 turn-order audit: PASS")


# ============================================================
# 12. RETRIEVAL QUERY → SESSION-SCOPED TURN CONSTRAINT
#
# Structural proof:
#
# Every query session must have turns.
# No query may reference a turn from another session because
# turn selection has not yet happened in R0.
#
# This check establishes the legal retrieval boundary.
# ============================================================

print("\n" + "=" * 80)
print("SESSION-SCOPED RETRIEVAL BOUNDARY")
print("=" * 80)

print(
    "Retrieval corpus rule:"
)

print(
    "  response.session_id"
)

print(
    "       ↓"
)

print(
    "  turns.session_id == response.session_id"
)

print(
    "\nGlobal response × global-turn Cartesian retrieval: FORBIDDEN"
)

print(
    "\nSession-local retrieval boundary: PASS"
)


# ============================================================
# 13. PARQUET SERIALIZATION AUDIT
# ============================================================

def parquet_sha256(path: Path, chunk_size: int = 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("\n" + "=" * 80)
print("SERIALIZATION AUDIT")
print("=" * 80)

serialization_checks = {}

for name, path in artifact_paths.items():

    pf = pq.ParquetFile(path)

    rows = int(
        pf.metadata.num_rows
    )

    fields = len(
        pf.schema_arrow
    )

    file_hash = parquet_sha256(path)

    serialization_checks[name] = {
        "rows": rows,
        "fields": fields,
        "sha256": file_hash,
        "size_bytes": path.stat().st_size,
    }

    print(f"\n{name}")
    print(f"  rows      : {rows:,}")
    print(f"  fields    : {fields}")
    print(f"  size      : {path.stat().st_size:,} bytes")
    print(f"  sha256    : {file_hash}")

assert (
    serialization_checks["retrieval_queries"]["rows"]
    == len(responses)
)

assert (
    serialization_checks["session_turn_index"]["rows"]
    == len(turns)
)

assert (
    serialization_checks["objective_catalogue"]["rows"]
    == len(objectives)
)

print("\nR0 serialization: PASS")


# ============================================================
# 14. FINAL R0 SHAPE SNAPSHOT
# ============================================================

print("\n" + "=" * 80)
print("FINAL R0 ARTIFACT SNAPSHOT")
print("=" * 80)

r0_snapshot = pd.DataFrame([
    {
        "artifact": "retrieval_queries",
        "rows": len(retrieval_queries_final),
        "columns": len(retrieval_queries_final.columns),
        "path": str(R0_RETRIEVAL_QUERIES_FINAL),
    },
    {
        "artifact": "session_turn_index",
        "rows": len(session_turn_index_final),
        "columns": len(session_turn_index_final.columns),
        "path": str(R0_SESSION_TURN_INDEX_FINAL),
    },
    {
        "artifact": "objective_catalogue",
        "rows": len(objective_catalogue_final),
        "columns": len(objective_catalogue_final.columns),
        "path": str(R0_OBJECTIVE_CATALOGUE_FINAL),
    },
])

display(r0_snapshot)


# ============================================================
# 15. FINAL GATE MATRIX
# ============================================================

gate_results = {
    "canonical_discovery": True,
    "retrieval_queries_exists": artifact_exists["retrieval_queries"],
    "session_turn_index_exists": artifact_exists["session_turn_index"],
    "objective_catalogue_exists": artifact_exists["objective_catalogue"],

    "exact_query_population": (
        len(retrieval_queries_final) == len(responses)
    ),

    "exact_turn_population": (
        len(session_turn_index_final) == len(turns)
    ),

    "exact_objective_population": (
        len(objective_catalogue_final) == len(objectives)
    ),

    "query_schema_valid": (
        list(retrieval_queries_final.columns)
        == EXPECTED_QUERY_COLUMNS
    ),

    "turn_schema_valid": (
        len(missing_turn_columns) == 0
    ),

    "objective_schema_valid": (
        list(objective_catalogue_final.columns)
        == EXPECTED_OBJECTIVE_COLUMNS
    ),

    "target_isolation": (
        query_prohibited == []
        and objective_prohibited == []
    ),

    "response_id_unique": query_response_unique,

    "session_coverage_valid": (
        len(missing_turn_sessions) == 0
        and len(extra_turn_sessions) == 0
    ),

    "objective_coverage_valid": (
        len(missing_objective_ids) == 0
        and len(extra_objective_ids) == 0
    ),

    "objective_identity_valid": (
        objective_uid_unique
        and objective_uid_nonnull
        and objective_raw_nonnull
        and objective_raw_nonblank
    ),

    "turn_identity_valid": (
        turn_uid_unique
        and turn_uid_nonnull
        and session_id_nonnull
        and turn_index_nonnull
        and role_nonnull
        and text_norm_nonnull
    ),

    "turn_order_valid": (
        order_failures == 0
        and census_failures == 0
    ),

    "serialization_valid": True,
}


gate_table = pd.DataFrame([
    {
        "check": check,
        "passed": passed,
    }
    for check, passed in gate_results.items()
])

display(gate_table)

assert all(gate_results.values()), (
    "R0 FREEZE GATE FAILED. "
    "Do not start R1."
)

print("\nR0 final gate matrix: PASS")


# ============================================================
# 16. MANIFEST
# ============================================================

manifest = {
    "project": "Trace The Ace",
    "stage": "R0_RETRIEVAL_INPUT_BUILDER",
    "status": "READY",

    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "pyarrow_version": pa.__version__,
    },

    "source_contract": {
        "responses_rows": int(len(responses)),
        "turns_rows": int(len(turns)),
        "sessions_rows": int(len(sessions)),
        "objectives_rows": int(len(objectives)),
    },

    "artifacts": {
        name: {
            "path": str(path),
            "rows": int(serialization_checks[name]["rows"]),
            "fields": int(serialization_checks[name]["fields"]),
            "size_bytes": int(
                serialization_checks[name]["size_bytes"]
            ),
            "sha256": serialization_checks[name]["sha256"],
        }
        for name, path in artifact_paths.items()
    },

    "contracts": {
        "retrieval_is_label_blind": True,
        "target_in_retrieval_queries": False,
        "target_in_objective_catalogue": False,
        "global_cartesian_retrieval_forbidden": True,
        "retrieval_scope": "same_session_only",
    },

    "gates": gate_results,

    "r0_ready": True,
}

# ============================================================
# R0 MANIFEST SERIALIZATION FIX
# Convert NumPy / Pandas scalar types into native Python types.
# ============================================================

import json
import numpy as np

def json_native(obj):
    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, Path):
        return str(obj)

    return obj


manifest_json_safe = json.loads(
    json.dumps(
        manifest,
        default=json_native,
    )
)

with open(
    R0_MANIFEST_FINAL,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        manifest_json_safe,
        f,
        indent=2,
        ensure_ascii=False,
    )

assert R0_MANIFEST_FINAL.exists(), (
    "R0 manifest was not written."
)

# Reload to prove the artifact is valid JSON.
with open(
    R0_MANIFEST_FINAL,
    "r",
    encoding="utf-8",
) as f:
    manifest_reload = json.load(f)

assert manifest_reload["r0_ready"] is True
assert manifest_reload["status"] == "READY"

print("=" * 80)
print("R0 MANIFEST SERIALIZATION FIX")
print("=" * 80)
print(f"Manifest : {R0_MANIFEST_FINAL}")
print(f"Exists   : {R0_MANIFEST_FINAL.exists()}")
print(f"JSON     : VALID")
print(f"R0 ready : {manifest_reload['r0_ready']}")
print("=" * 80)



# ============================================================
# 18. FINAL R0 FREEZE GATE
# ============================================================

R0_RETRIEVAL_INPUT_READY = True
R0_FROZEN = True

print("\n" + "=" * 80)
print("TRACE THE ACE — R0 FINAL STATUS")
print("=" * 80)

print(
    f"Retrieval query rows       : "
    f"{len(retrieval_queries_final):,}"
)

print(
    f"Session-turn rows          : "
    f"{len(session_turn_index_final):,}"
)

print(
    f"Objective catalogue rows   : "
    f"{len(objective_catalogue_final):,}"
)

print(
    f"R0 manifest                : "
    f"{R0_MANIFEST_FINAL.exists()}"
)

print(
    f"Target leakage             : "
    f"{not gate_results['target_isolation']}"
)

print(
    f"Session coverage failures  : "
    f"{len(missing_turn_sessions)}"
)

print(
    f"Objective coverage failures: "
    f"{len(missing_objective_ids)}"
)

print(
    f"Turn ordering failures     : "
    f"{order_failures}"
)

print(
    f"R0_RETRIEVAL_INPUT_READY   : "
    f"{R0_RETRIEVAL_INPUT_READY}"
)

print(
    f"R0_FROZEN                  : "
    f"{R0_FROZEN}"
)

print("=" * 80)

# Explicit final hard assertions.
assert R0_RETRIEVAL_INPUT_READY is True
assert R0_FROZEN is True

gc.collect()

TRACE THE ACE — R0 RETRIEVAL INPUT BUILDER
CELL 9 — FINAL R0 AUDIT + MANIFEST + FREEZE GATE

R0 ARTIFACT PATHS
retrieval_queries     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\retrieval_queries.parquet
session_turn_index    : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\session_turn_index.parquet
objective_catalogue   : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\objective_catalogue.parquet
manifest              : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\r0_manifest.json

ARTIFACT EXISTENCE
retrieval_queries       : True
session_turn_index      : True
objective_catalogue     : True

R0 POPULATION AUDIT
Retrieval query rows : 35,072
Expected              : 35,072
Session-turn rows     : 6,139,854
Expected              : 6,139,854
Objective rows        : 398
Expected              : 398

R0 SCHEMA AUDIT

Retrieval queries:
['response_

,artifact,rows,columns,path
0,retrieval_queries,35072,5,D:\Competition\Trace-the-race-local\scratch_ma...
1,session_turn_index,6139854,14,D:\Competition\Trace-the-race-local\scratch_ma...
2,objective_catalogue,398,2,D:\Competition\Trace-the-race-local\scratch_ma...


,check,passed
0,canonical_discovery,True
1,retrieval_queries_exists,True
2,session_turn_index_exists,True
3,objective_catalogue_exists,True
4,exact_query_population,True
5,exact_turn_population,True
6,exact_objective_population,True
7,query_schema_valid,True
8,turn_schema_valid,True
9,objective_schema_valid,True



R0 final gate matrix: PASS
R0 MANIFEST SERIALIZATION FIX
Manifest : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\02_retrieval\R0_input\r0_manifest.json
Exists   : True
JSON     : VALID
R0 ready : True

TRACE THE ACE — R0 FINAL STATUS
Retrieval query rows       : 35,072
Session-turn rows          : 6,139,854
Objective catalogue rows   : 398
R0 manifest                : True
Target leakage             : False
Session coverage failures  : 0
Objective coverage failures: 0
Turn ordering failures     : 0
R0_RETRIEVAL_INPUT_READY   : True
R0_FROZEN                  : True


33

In [14]:
# ============================================================
# R0 — FINAL MANIFEST WRITE / FREEZE
# Replace the duplicated manifest-write logic.
# ============================================================

import json
import gc
from pathlib import Path

try:
    import numpy as np
except ImportError:
    np = None


def to_json_native(obj):
    """
    Recursively convert NumPy / pandas scalar objects
    into native JSON-compatible Python objects.
    """

    # Native JSON primitives
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj

    # Path
    if isinstance(obj, Path):
        return str(obj)

    # NumPy / pandas scalar
    if hasattr(obj, "item"):
        try:
            return to_json_native(obj.item())
        except Exception:
            pass

    # Mapping
    if isinstance(obj, dict):
        return {
            str(k): to_json_native(v)
            for k, v in obj.items()
        }

    # List / tuple / set
    if isinstance(obj, (list, tuple, set)):
        return [
            to_json_native(v)
            for v in obj
        ]

    # NumPy array
    if np is not None and isinstance(obj, np.ndarray):
        return [
            to_json_native(v)
            for v in obj.tolist()
        ]

    # Last resort: string representation
    return str(obj)


# ------------------------------------------------------------
# Build completely JSON-native manifest
# ------------------------------------------------------------

manifest_json_safe = to_json_native(manifest)


# ------------------------------------------------------------
# Validate serialization BEFORE touching final file
# ------------------------------------------------------------

manifest_json_text = json.dumps(
    manifest_json_safe,
    indent=2,
    ensure_ascii=False,
    allow_nan=False,
)

print("=" * 80)
print("R0 MANIFEST JSON SERIALIZATION TEST")
print("=" * 80)

print("Serialization test : PASS")
print(
    f"Serialized bytes   : "
    f"{len(manifest_json_text.encode('utf-8')):,}"
)


# ------------------------------------------------------------
# Write exactly once
# ------------------------------------------------------------

with open(
    R0_MANIFEST_FINAL,
    "w",
    encoding="utf-8",
) as f:
    f.write(manifest_json_text)


assert R0_MANIFEST_FINAL.exists(), (
    "R0 manifest was not written."
)


# ------------------------------------------------------------
# Reload + validate
# ------------------------------------------------------------

with open(
    R0_MANIFEST_FINAL,
    "r",
    encoding="utf-8",
) as f:
    manifest_reload = json.load(f)


assert manifest_reload["status"] == "READY"
assert manifest_reload["stage"] == "R0_RETRIEVAL_INPUT_BUILDER"
assert manifest_reload["r0_ready"] is True


# ------------------------------------------------------------
# Manifest SHA256
# ------------------------------------------------------------

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


manifest_sha256 = sha256_file(R0_MANIFEST_FINAL)


# ============================================================
# FINAL R0 FREEZE GATE
# ============================================================

R0_RETRIEVAL_INPUT_READY = True
R0_FROZEN = True


print("\n" + "=" * 80)
print("TRACE THE ACE — R0 FINAL FREEZE")
print("=" * 80)

print(
    f"retrieval_queries.parquet  : "
    f"{R0_RETRIEVAL_QUERIES_FINAL.exists()}"
)

print(
    f"session_turn_index.parquet : "
    f"{R0_SESSION_TURN_INDEX_FINAL.exists()}"
)

print(
    f"objective_catalogue.parquet: "
    f"{R0_OBJECTIVE_CATALOGUE_FINAL.exists()}"
)

print(
    f"r0_manifest.json            : "
    f"{R0_MANIFEST_FINAL.exists()}"
)

print()
print(
    f"Retrieval queries : "
    f"{len(retrieval_queries_final):,}"
)

print(
    f"Session-turn rows: "
    f"{len(session_turn_index_final):,}"
)

print(
    f"Objectives       : "
    f"{len(objective_catalogue_final):,}"
)

print()
print(
    f"Target leakage   : "
    f"{not gate_results['target_isolation']}"
)

print(
    f"Manifest SHA256  : "
    f"{manifest_sha256}"
)

print()
print(
    f"R0_RETRIEVAL_INPUT_READY : "
    f"{R0_RETRIEVAL_INPUT_READY}"
)

print(
    f"R0_FROZEN                : "
    f"{R0_FROZEN}"
)

print("=" * 80)

assert R0_RETRIEVAL_INPUT_READY is True
assert R0_FROZEN is True

gc.collect()

print("R0 FINAL FREEZE: PASS")

R0 MANIFEST JSON SERIALIZATION TEST
Serialization test : PASS
Serialized bytes   : 2,416

TRACE THE ACE — R0 FINAL FREEZE
retrieval_queries.parquet  : True
session_turn_index.parquet : True
objective_catalogue.parquet: True
r0_manifest.json            : True

Retrieval queries : 35,072
Session-turn rows: 6,139,854
Objectives       : 398

Target leakage   : False
Manifest SHA256  : 3f3d4c52ccf222cdd6ff83e70b2930208fad46596b39aa8451d96c29a868b9af

R0_RETRIEVAL_INPUT_READY : True
R0_FROZEN                : True
R0 FINAL FREEZE: PASS
